# Final TopiOCQA Retrieval Evaluation

This notebook evaluates the locked ROCC systems on the complete TopiOCQA development split with BM25 and ANCE, comparing Viterbi, FFBS, retrieval fusion, and budget effects.

The code names `pretrained`, `imitation_control_e6`, and `bm25_treatment_e6` correspond to the thesis checkpoints $f_{\mathrm{base}}$, $f_{\mathrm{imit}}$, and $f_{\mathrm{BM25}}$. The base student has three training epochs. Both post-trained students start from that base and receive six additional epochs.

At budget $B$, `I`, `R`, and `D` denote $I_B$, $R_B$, and $D_B$. `I+R+D` denotes $IRD_B$, not an arithmetic sum of scores. Their rankings are $L_I^B$, $L_R^B$, $L_D^B$, and $L_{IRD}^B$. `RRF10` means reciprocal rank fusion with $\kappa=10$. `IterCQR-G+2S-RRF10` is the thesis comparison $G+2S$.

### Evaluation setup

The evaluation supports local BM25 and GPU-based ANCE execution, locally or in Colab. Prepare the checkout and dependencies first; `ROCC_ROOT` may point to a local or mounted Drive checkout. Checkpoints, indices, caches and results use this checkout by default; the existing artifact-root settings can point to persistent storage.

Use `NB07A_BACKEND` in the control cell below to select the backend. `None` selects ANCE in Colab and BM25 locally; `"ance"` or `"bm25"` selects a backend explicitly. Completed backend results are restored without rerunning query generation, retrieval, fusion, or bootstrap. Fresh ANCE execution requires the complete local or Drive-linked index and sufficient GPU memory; the original float32 capacity check remains active.

In [1]:
# None selects automatically: ANCE in Colab, BM25 locally.
# Select "ance" or "bm25" explicitly for that backend.
NB07A_BACKEND = None

In [2]:
import gc
import gzip
import hashlib
import importlib.metadata
import inspect
import json
import math
import os
import platform
import random
import re
import shutil
import subprocess
import sys
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence


# Use the already prepared checkout (ROCC_ROOT may point to a mounted Drive checkout).
def is_colab_runtime() -> bool:
    try:
        import google.colab  # noqa: F401
    except ModuleNotFoundError:
        return False
    return True

IN_COLAB = is_colab_runtime()


def resolve_project_root(start: Path) -> Path:
    configured = os.environ.get("ROCC_ROOT") or os.environ.get("MASTER_THESIS_PROJECT_ROOT")
    if configured:
        candidate = Path(configured).expanduser().resolve()
        if not (candidate / "experiments" / "rocc").is_dir():
            raise FileNotFoundError(f"Configured ROCC checkout is missing: {candidate}")
        return candidate
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "rocc").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from your ROCC checkout, or set ROCC_ROOT to it. "
        "In Colab, first clone the public repository over HTTPS or authenticate "
        "your own private checkout; no private clone or deploy key is assumed."
    )


PROJECT_ROOT = resolve_project_root(Path.cwd())
os.environ["MASTER_THESIS_PROJECT_ROOT"] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from tqdm.auto import tqdm
from transformers import AutoTokenizer

from experiments.rocc import (
    CachedIterCQRBM25Config,
    CachedIterCQRBM25Pipeline,
    HistorySelectorConfig,
    ORACLE_GATE_DEPTH_BIN_ORDER,
    ORACLE_GATE_DEPTH_BOUNDS,
    SelectorExperimentConfig,
    TOPIOCQA_CORPUS_PASSAGES,
    add_conversation_columns,
    build_topiocqa_conversation_samples,
    check_dense_gpu_capacity,
    derive_query_seed,
    ensure_colab_retrieval_dependencies,
    evaluate_fixed_rrf_views,
    inspect_bm25_lucene_index,
    load_bm25_retriever,
    load_crf_inference_model,
    load_dense_ance_torch_retriever,
    load_itercqr_rewriter,
    load_itercqr_tokenizer,
    load_or_compute_current_query_only,
    load_topiocqa_frame,
    materialize_history_arm_inputs,
    materialize_selector_direct_queries,
    normalize_rewrite,
    paired_bootstrap_mean_ci,
    predict_crf_history_candidates,
    prepare_colab_itercqr_runtime,
    read_selected_histories_jsonl,
    resolve_topiocqa_resources,
    run_query_retrieval_pipeline,
    serialize_itercqr_input,
    selector_parameter_counts,
    summarize_dense_faiss_shards,
    validate_itercqr_checkpoint,
    write_selected_histories_jsonl,
)


DEFAULT_BACKEND = "ance" if IN_COLAB else "bm25"
BACKEND = os.environ.get(
    "ROCC_NB07_BACKEND",
    str(globals().get("NB07A_BACKEND") or DEFAULT_BACKEND),
).strip().lower()
if BACKEND not in {"bm25", "ance"}:
    raise ValueError("ROCC_NB07_BACKEND must be bm25 or ance.")

SEED = 13
BUDGETS = (64, 128, 256, 512)
PRIMARY_BUDGET = 64
D_NO_HISTORY_QUERY_RULE = "dataset_current_query_without_t5"
RETRIEVAL_TOP_K = 1_000
EVAL_KS = (3, 10, 100, 1000)
RRF_K = 10
BOOTSTRAP_REPLICATES = 10_000
ITER_SAMPLING_SEED = 13
ITER_SAMPLING_TEMPERATURE = 1.0
ITER_SAMPLING_TOP_K = 50
ITER_SAMPLING_TOP_P = 1.0
CRF_SAMPLING_SEED = 13
SAMPLING_SEED_SCHEME = "sha256_first_8_bytes_big_endian_v1"
NUM_ITER_SAMPLES = 2
NUM_FFBS_SAMPLES = 2
SELECTOR_BATCH_SIZE = 32
REWRITE_BATCH_SIZE = 16
RETRIEVAL_BATCH_SIZE = 64
SYSTEM_ORDER = (
    "pretrained",
    "imitation_control_e6",
    "bm25_treatment_e6",
)
METRIC_ORDER = ("MRR", "nDCG@3", "R@10", "R@100", "R@1000")
PRIMARY_COMPARISON_ORDER = (
    "pretrained_ird_minus_itercqr1",
    "pretrained_ird_minus_itercqr_g2s",
    "treatment_ird_minus_control_ird",
    "treatment_ird_minus_pretrained_ird",
    "control_ird_minus_pretrained_ird",
    "treatment_r_minus_control_r",
    "treatment_r_minus_pretrained_r",
    "control_r_minus_pretrained_r",
    "treatment_d_minus_control_d",
    "treatment_d_minus_pretrained_d",
    "control_d_minus_pretrained_d",
    "pretrained_ird_b64_minus_itercqr_g2s_b512",
)

NB05_PRETRAINED_SHA256 = (
    "a07271abefbc37ef08dba4eb94adb0e165f53a652060c069c909a7a1614115c4"
)
NB06_CONTROL_E6_SHA256 = (
    "c754a91ac71111b82ca067d69490f29e7e6295fc368e4e5a3075e8866480dca6"
)
NB06_TREATMENT_E6_SHA256 = (
    "a8d00d1cdcc7116e4f20833429a1acb879b2346bea4840319b555c073410992d"
)
NB05_TRAINING_RECIPE_VERSION = "independent_hf_full_v2"
POST_TRAINING_RECIPE_VERSION = "hard_relabel_nltk_nostop_controlled_ce_crf_v2"

DEFAULT_ARTIFACT_ROOT = (
    PROJECT_ROOT
)
ARTIFACT_PROJECT_ROOT = Path(
    os.environ.get(
        "ROCC_NB07_ARTIFACT_ROOT",
        str(DEFAULT_ARTIFACT_ROOT),
    )
).expanduser().resolve()
if IN_COLAB and not ARTIFACT_PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        "NB07a artifact directory is missing from Google Drive: "
        f"{ARTIFACT_PROJECT_ROOT}"
    )
RESULT_DIR = Path(
    os.environ.get(
        "ROCC_NB07_RESULT_DIR",
        str(
            ARTIFACT_PROJECT_ROOT
            / "experiments/results/07a_topiocqa_final"
        ),
    )
).expanduser().resolve()
RESULT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = ARTIFACT_PROJECT_ROOT / "experiments/data/topiocqa"
ITERCQR_MODEL_DIR = (
    PROJECT_ROOT / "experiments/model/IterCQR/IterCQR Model"
)
BM25_INDEX_DIR = (
    PROJECT_ROOT
    / "experiments/data/pyserini_bm25_lucene_topiocqa/lucene_index"
)
DENSE_INDEX_DIR = (
    PROJECT_ROOT
    / "experiments/data/pyserini_ance_faiss_topiocqa"
    / "faiss_flat_index_full_sharded"
)
PIPELINE_CACHE_DB = Path(
    os.environ.get(
        "ROCC_NB07_PIPELINE_CACHE_DB",
        str(
            PROJECT_ROOT
            / ".cache/master_thesis/03_oracle_headroom_analysis"
            / "runtime_cache.sqlite3"
        ),
    )
).expanduser()


def artifact_path(relative: str) -> Path:
    return ARTIFACT_PROJECT_ROOT / relative


PRETRAINED_CHECKPOINT = artifact_path(
    "experiments/results/05_history_selector/full_candidates"
    "/collapsed_crf/model.pt"
)
PRETRAINED_MANIFEST = PRETRAINED_CHECKPOINT.parent / "manifest.json"
POST_ROOT = artifact_path(
    "experiments/results/06_post_training"
    "/controlled_hard_relabel_nostop_e6"
)
CONTROL_CHECKPOINT = (
    POST_ROOT
    / "training/imitation_control/checkpoint_epoch_6.pt"
)
TREATMENT_CHECKPOINT = (
    POST_ROOT
    / "training/bm25_treatment/checkpoint_epoch_6.pt"
)
CHECKPOINTS = {
    "pretrained": PRETRAINED_CHECKPOINT,
    "imitation_control_e6": CONTROL_CHECKPOINT,
    "bm25_treatment_e6": TREATMENT_CHECKPOINT,
}
EXPECTED_CHECKPOINT_HASHES = {
    "pretrained": NB05_PRETRAINED_SHA256,
    "imitation_control_e6": NB06_CONTROL_E6_SHA256,
    "bm25_treatment_e6": NB06_TREATMENT_E6_SHA256,
}
ARCHITECTURE_DECISION_PATH = artifact_path(
    "experiments/results/05_history_selector/train600"
    "/architecture/decision.json"
)
FUSION_DECISION_PATH = artifact_path(
    "experiments/results/05_history_selector/train600"
    "/fusion/decision.json"
)
QUERY_BUNDLE_PATH = RESULT_DIR / "query_bundle/queries.jsonl"
QUERY_BUNDLE_MANIFEST_PATH = (
    RESULT_DIR / "query_bundle/manifest.json"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": "#39424e",
        "axes.grid": True,
        "grid.color": "#d7dde5",
        "grid.linewidth": 0.7,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 10,
    }
)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def canonical_json(value: Any) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )


def write_json(path: Path, value: Mapping[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, default=str)
        + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def write_frame(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp")
    frame.to_csv(
        temporary,
        index=False,
        float_format="%.17g",
        lineterminator="\n",
    )
    temporary.replace(path)


def frame_summary(
    frame: pd.DataFrame,
    group_columns: Sequence[str],
) -> pd.DataFrame:
    return (
        frame.groupby(list(group_columns), observed=True)
        .agg(
            n=("sample_id", "nunique"),
            **{metric: (metric, "mean") for metric in METRIC_ORDER},
        )
        .reset_index()
    )


def query_id_for(row: Mapping[str, Any]) -> str:
    payload = "\0".join(
        [
            str(row["family"]),
            str(row["system"]),
            str(row["sample_id"]),
            str(row["budget"]),
            str(row["view"]),
            str(row.get("candidate_index", 0)),
            str(row["query_norm"]),
        ]
    )
    return sha256_text(payload)


def history_as_dicts(sample: Any) -> list[dict[str, Any]]:
    return [
        {
            "turn_id": int(turn.turn_id),
            "question": str(turn.question),
            "answer": str(turn.answer),
        }
        for turn in sample.history
    ]


def word_count(text: str) -> int:
    return len(re.findall(r"\S+", str(text)))


def history_word_count(history: Sequence[Mapping[str, Any]]) -> int:
    return sum(
        word_count(turn.get("question", ""))
        + word_count(turn.get("answer", ""))
        for turn in history
    )


if IN_COLAB:
    colab_runtime = prepare_colab_itercqr_runtime()
    ITERCQR_MODEL_DIR = colab_runtime.checkpoint.local_checkpoint
    display(pd.DataFrame([colab_runtime.corpus.to_dict()]))
    display(pd.DataFrame([colab_runtime.checkpoint.to_dict()]))

SELECTOR_CONFIG = HistorySelectorConfig(
    model_name="sentence-transformers/all-MiniLM-L12-v2",
    max_length=512,
    history_order="recent_first",
    batch_size=SELECTOR_BATCH_SIZE,
    seed=SEED,
)
PIPELINE_CONFIG = CachedIterCQRBM25Config(
    cache_db=PIPELINE_CACHE_DB,
    model_dir=ITERCQR_MODEL_DIR,
    bm25_index_dir=BM25_INDEX_DIR,
    device=str(DEVICE),
    rewrite_batch_size=REWRITE_BATCH_SIZE,
    retrieval_batch_size=RETRIEVAL_BATCH_SIZE,
    retrieval_workers=max(
        1,
        min(8, (os.cpu_count() or 2) // 2),
    ),
    retrieval_top_k=RETRIEVAL_TOP_K,
    eval_ks=EVAL_KS,
    progress=True,
)
EXPERIMENT_CONFIG = SelectorExperimentConfig(
    train_labels=artifact_path(
        "experiments/results/04_teacher"
        "/rocc_history_labels_topiocqa_train.jsonl"
    ),
    train_manifest=artifact_path(
        "experiments/results/04_teacher"
        "/rocc_history_labels_topiocqa_train.manifest.json"
    ),
    topiocqa_data_dir=DATA_DIR,
    result_dir=RESULT_DIR,
    encoding_cache_dir=PROJECT_ROOT
    / ".cache/master_thesis/07a_topiocqa_final",
    itercqr_model_dir=ITERCQR_MODEL_DIR,
    bm25_index_dir=BM25_INDEX_DIR,
    pipeline_cache_db=PIPELINE_CACHE_DB,
    seed=SEED,
    budgets=BUDGETS,
    rewrite_batch_size=REWRITE_BATCH_SIZE,
    retrieval_batch_size=RETRIEVAL_BATCH_SIZE,
    retrieval_workers=PIPELINE_CONFIG.retrieval_workers,
    retrieval_top_k=RETRIEVAL_TOP_K,
    eval_ks=EVAL_KS,
    bootstrap_replicates=BOOTSTRAP_REPLICATES,
)

display(
    pd.DataFrame(
        [
            {
                "backend": BACKEND,
                "device": str(DEVICE),
                "project_root": str(PROJECT_ROOT),
                "artifact_root": str(ARTIFACT_PROJECT_ROOT),
                "result_dir": str(RESULT_DIR),
                "budgets": BUDGETS,
                "primary_budget": PRIMARY_BUDGET,
                "retrieval_top_k": RETRIEVAL_TOP_K,
            }
        ]
    )
)


,backend,device,project_root,artifact_root,result_dir,budgets,primary_budget,retrieval_top_k
0,ance,cuda:0,,,/experiments/...,"(64, 128, 256, 512)",64,1000


### Validating the frozen evaluation protocol

Each selector checkpoint is matched byte-for-byte against its expected SHA-256 hash. The associated manifests then verify the experimental meaning of these files. The NB05 checkpoint must use collapsed BIO labels, a CRF decoder, and the fixed CE-plus-CRF objective. Both NB06 checkpoints must represent complete six-epoch runs initialized from that exact NB05 checkpoint with the registered control or treatment targets.

The locked NB05 decisions are also restored. The selected architecture must be `collapsed_crf` at B64, while the selected fusion must be RRF10 over the `I`, `R`, and `D` views.

The IterCQR checkpoint is validated separately. Python, library, CUDA, device, and hardware information is recorded in a software manifest when the evaluation environment is first created.

In [3]:
# Only explicit local mode binds newly trained checkpoints to their manifests.
from experiments.rocc.paths import bind_notebook_lineage
EXPECTED_CHECKPOINT_HASHES = bind_notebook_lineage(
    RESULT_DIR, "checkpoint_inputs", EXPECTED_CHECKPOINT_HASHES, CHECKPOINTS,
    manifest_fields=(
        ("pretrained", PRETRAINED_CHECKPOINT.parent / "manifest.json", ("checkpoint_sha256",)),
        ("pretrained", ARCHITECTURE_DECISION_PATH, ("selector_checkpoint_sha256",)),
        ("imitation_control_e6", CONTROL_CHECKPOINT.parent / "manifest.json", ("epoch_checkpoints", "6", "sha256")),
        ("bm25_treatment_e6", TREATMENT_CHECKPOINT.parent / "manifest.json", ("epoch_checkpoints", "6", "sha256")),
    ),
)
NB05_PRETRAINED_SHA256 = EXPECTED_CHECKPOINT_HASHES["pretrained"]
NB06_CONTROL_E6_SHA256 = EXPECTED_CHECKPOINT_HASHES["imitation_control_e6"]
NB06_TREATMENT_E6_SHA256 = EXPECTED_CHECKPOINT_HASHES["bm25_treatment_e6"]

for system, checkpoint in CHECKPOINTS.items():
    if not checkpoint.exists():
        raise FileNotFoundError(checkpoint)
    observed = sha256_file(checkpoint)
    expected = EXPECTED_CHECKPOINT_HASHES[system]
    if observed != expected:
        raise RuntimeError(
            f"Checkpoint hash differs for {system}: {observed}"
        )

pretrained_manifest = json.loads(
    PRETRAINED_MANIFEST.read_text(encoding="utf-8")
)
pretrained_config = pretrained_manifest["config"]
expected_nb05_fields = {
    "training_recipe_version": NB05_TRAINING_RECIPE_VERSION,
    "label_space": "collapsed",
    "decoder": "crf",
    "token_loss_weight": 1.0,
    "crf_loss_weight": 1.0,
}
for field, expected in expected_nb05_fields.items():
    if pretrained_config.get(field) != expected:
        raise RuntimeError(
            f"NB05 manifest field {field} differs: "
            f"{pretrained_config.get(field)!r}"
        )

for arm, checkpoint in (
    ("imitation_control", CONTROL_CHECKPOINT),
    ("bm25_treatment", TREATMENT_CHECKPOINT),
):
    manifest = json.loads(
        (checkpoint.parent / "manifest.json").read_text(
            encoding="utf-8"
        )
    )
    config = manifest["config"]
    if not manifest.get("complete"):
        raise RuntimeError(f"NB06 run is incomplete: {arm}")
    required = {
        "arm": arm,
        "epochs": 6,
        "start_checkpoint_sha256": NB05_PRETRAINED_SHA256,
        "training_recipe_version": POST_TRAINING_RECIPE_VERSION,
        "loss": "weighted_token_CE_plus_token_normalized_CRF_NLL",
        "token_loss_weight": 1.0,
        "crf_loss_weight": 1.0,
    }
    for field, expected in required.items():
        if config.get(field) != expected:
            raise RuntimeError(
                f"NB06 {arm}: {field} differs: "
                f"{config.get(field)!r}"
            )

architecture_decision = json.loads(
    ARCHITECTURE_DECISION_PATH.read_text(encoding="utf-8")
)
fusion_decision = json.loads(
    FUSION_DECISION_PATH.read_text(encoding="utf-8")
)
if not (
    architecture_decision.get("locked") is True
    and architecture_decision.get("selected_arm") == "collapsed_crf"
    and int(architecture_decision.get("budget")) == PRIMARY_BUDGET
    and architecture_decision.get("selector_checkpoint_sha256")
    == NB05_PRETRAINED_SHA256
):
    raise RuntimeError("The frozen NB05 architecture differs.")
if not (
    fusion_decision.get("locked") is True
    and fusion_decision.get("selected_method") == "rrf10"
    and int(fusion_decision.get("budget")) == PRIMARY_BUDGET
    and fusion_decision.get("views") == ["I", "R", "D"]
):
    raise RuntimeError("The frozen NB05 fusion differs.")

validate_itercqr_checkpoint(ITERCQR_MODEL_DIR)
software_manifest = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "transformers": importlib.metadata.version("transformers"),
    "pytorch_crf": importlib.metadata.version("pytorch-crf"),
    "cuda_available": torch.cuda.is_available(),
    "cuda_device": (
        torch.cuda.get_device_name(DEVICE)
        if DEVICE.type == "cuda"
        else None
    ),
    "cpu_count": os.cpu_count(),
}
software_manifest_path = RESULT_DIR / "protocol/software.json"
if not software_manifest_path.exists():
    write_json(software_manifest_path, software_manifest)
display(
    pd.DataFrame(
        [
            {
                "system": system,
                "checkpoint": str(CHECKPOINTS[system]),
                "sha256": EXPECTED_CHECKPOINT_HASHES[system],
            }
            for system in SYSTEM_ORDER
        ]
    )
)
display(pd.DataFrame([software_manifest]))


,system,checkpoint,sha256
0,pretrained,/experiments/...,a07271abefbc37ef08dba4eb94adb0e165f53a652060c0...
1,imitation_control_e6,/experiments/...,c754a91ac71111b82ca067d69490f29e7e6295fc368e4e...
2,bm25_treatment_e6,/experiments/...,a8d00d1cdcc7116e4f20833429a1acb879b2346bea4840...


,python,platform,torch,transformers,pytorch_crf,cuda_available,cuda_device,cpu_count
0,3.12.3,Linux-6.17.0-124040-tuxedo-x86_64-with-glibc2.39,2.11.0+cu130,4.57.3,0.7.2,True,NVIDIA GeForce RTX 4070 Laptop GPU,32


All three selector checkpoints match their registered SHA-256 hashes and passed the corresponding manifest checks. The evaluation therefore uses the frozen NB05 pretrained model and the intended epoch-6 NB06 control and treatment models.

The software table above describes the current notebook or restore environment, not necessarily the machine that originally produced the selected rankings. The recorded BM25 retrieval run used Python 3.12.3, PyTorch 2.11.0 with CUDA 13.0, Transformers 4.57.3, and an NVIDIA RTX 4070 Laptop GPU. The recorded ANCE retrieval run used Python 3.12.13, PyTorch 2.11.0 with CUDA 12.8, Transformers 5.13.1, and an NVIDIA RTX PRO 6000 Blackwell Server Edition. The backend-specific `runtime.json` files are authoritative for retrieval hardware and software.

### Freezing the final evaluation

All 2,514 TopiOCQA development queries are reconstructed and aligned with their histories and gold passages. The expected depth distribution and a SHA-256 hash over the sorted sample IDs fix the evaluation population across all runs.

The query bundle is reused only when its file hash, population, checkpoints, budgets, row count, and sampling configuration match the frozen protocol. This avoids regenerating expensive selector and IterCQR outputs while ensuring that BM25 and ANCE evaluate identical queries.

A completed backend analysis is reused only when its protocol identity and registered result-file hashes remain valid. `selection_performed=False` confirms that the development split was used exclusively for final evaluation. These checks prevent stale, partial, or mismatched artifacts from affecting the reported results.

In [4]:
resources = resolve_topiocqa_resources(DATA_DIR)
dev_frame = add_conversation_columns(
    load_topiocqa_frame(resources, splits=("dev",))
)
dev_samples = build_topiocqa_conversation_samples(
    dev_frame,
    minimum_history_depth=0,
    progress=False,
)
sample_by_id = {
    str(sample.sample_id): sample for sample in dev_samples
}
sample_ids = [str(sample.sample_id) for sample in dev_samples]
if len(sample_ids) != len(set(sample_ids)):
    raise RuntimeError("Duplicate sample IDs in TopiOCQA dev.")

depth_by_sample = {
    str(sample.sample_id): len(sample.history)
    for sample in dev_samples
}
depth_counts = pd.Series(depth_by_sample).value_counts()
population_counts = {
    "all": len(dev_samples),
    "history_0": int(depth_counts.get(0, 0)),
    "history_1": int(depth_counts.get(1, 0)),
    "history_ge_2": int(
        sum(depth >= 2 for depth in depth_by_sample.values())
    ),
}
if population_counts != {
    "all": 2_514,
    "history_0": 205,
    "history_1": 205,
    "history_ge_2": 2_104,
}:
    raise RuntimeError(
        f"Unexpected development population: {population_counts}"
    )

dev_rows = [
    {
        "sample_id": str(sample.sample_id),
        "split": "dev",
        "conv_id": int(sample.conv_id),
        "turn_id": int(sample.turn_id),
        "history_len": len(sample.history),
        "current_query": str(sample.current_query),
        "history": history_as_dicts(sample),
    }
    for sample in dev_samples
]
row_by_id = {str(row["sample_id"]): row for row in dev_rows}
gold_by_sample = {
    str(row.sample_id): [
        str(docid) for docid in row.positive_ctx_passage_ids
    ]
    for row in dev_frame.itertuples(index=False)
}
if set(gold_by_sample) != set(sample_ids):
    raise RuntimeError("Gold labels and development samples are not aligned.")

population_sha256 = sha256_text("\n".join(sorted(sample_ids)))
population_manifest = {
    "dataset": "TopiOCQA dev",
    "queries": len(sample_ids),
    "counts": population_counts,
    "sample_id_sha256": population_sha256,
}
write_json(
    RESULT_DIR / "protocol/population.json",
    population_manifest,
)
display(pd.DataFrame([population_manifest]))


bundle_file_exists = QUERY_BUNDLE_PATH.is_file()
bundle_manifest_exists = QUERY_BUNDLE_MANIFEST_PATH.is_file()
if bundle_file_exists != bundle_manifest_exists:
    raise RuntimeError(
        "Query bundle and manifest are only partially present."
    )

REUSE_FROZEN_BUNDLE = bundle_file_exists and bundle_manifest_exists
frozen_bundle_manifest: dict[str, Any] | None = None
bundle_sha256: str | None = None

if REUSE_FROZEN_BUNDLE:
    frozen_bundle_manifest = json.loads(
        QUERY_BUNDLE_MANIFEST_PATH.read_text(encoding="utf-8")
    )
    bundle_sha256 = sha256_file(QUERY_BUNDLE_PATH)
    expected_bundle_identity = {
        "schema_version": 1,
        "protocol": "nb07a_topiocqa_final_v2",
        "d_no_history_query_rule": D_NO_HISTORY_QUERY_RULE,
        "population_sha256": population_sha256,
        "query_bundle_sha256": bundle_sha256,
        "rows": 143_298,
        "budgets": list(BUDGETS),
        "checkpoint_sha256": EXPECTED_CHECKPOINT_HASHES,
        "sampling_seed_scheme": SAMPLING_SEED_SCHEME,
        "iter_sampling_seed": ITER_SAMPLING_SEED,
        "iter_sampling_temperature": ITER_SAMPLING_TEMPERATURE,
        "iter_sampling_top_k": ITER_SAMPLING_TOP_K,
        "iter_sampling_top_p": ITER_SAMPLING_TOP_P,
        "crf_sampling_seed": CRF_SAMPLING_SEED,
        "complete": True,
        "selection_performed": False,
    }
    for field, expected in expected_bundle_identity.items():
        observed = frozen_bundle_manifest.get(field)
        if observed != expected:
            raise RuntimeError(
                f"Query-bundle identity differs at {field}: {observed!r}"
            )

backend_dir = RESULT_DIR / "backends" / BACKEND
backend_manifest_path = backend_dir / "manifest.json"
existing_final_manifest: dict[str, Any] | None = None
REUSE_COMPLETE_ANALYSIS = False

if REUSE_FROZEN_BUNDLE and backend_manifest_path.is_file():
    candidate_manifest = json.loads(
        backend_manifest_path.read_text(encoding="utf-8")
    )
    expected_analysis_identity = {
        "schema_version": 1,
        "protocol": "nb07a_topiocqa_final_v2",
        "d_no_history_query_rule": D_NO_HISTORY_QUERY_RULE,
        "backend": BACKEND,
        "population_sha256": population_sha256,
        "population_queries": len(sample_ids),
        "query_bundle_sha256": bundle_sha256,
        "checkpoint_sha256": EXPECTED_CHECKPOINT_HASHES,
        "budgets": list(BUDGETS),
        "primary_budget": PRIMARY_BUDGET,
        "rrf_k": RRF_K,
        "retrieval_top_k": RETRIEVAL_TOP_K,
        "bootstrap_replicates": BOOTSTRAP_REPLICATES,
        "primary_comparison_order": list(PRIMARY_COMPARISON_ORDER),
        "complete": True,
        "selection_performed": False,
    }
    identity_matches = all(
        candidate_manifest.get(field) == expected
        for field, expected in expected_analysis_identity.items()
    )
    recorded_hashes = candidate_manifest.get("files", {})
    budget_comparison_file = (
        "budget_mrr_comparisons.csv"
        if "budget_mrr_comparisons.csv" in recorded_hashes
        else "saturation_comparisons.csv"
    )
    core_analysis_files = (
        "route_metrics_by_query.csv",
        "route_summary.csv",
        "primary_comparisons.csv",
        "crf_decoder_metrics_by_query.csv",
        "crf_decoder_summary.csv",
        "crf_decoder_comparisons.csv",
        "compression_summary.csv",
        "budget_identity.csv",
        budget_comparison_file,
        "depth_strata_summary.csv",
    )
    recorded_projection = {
        key: value
        for key, value in recorded_hashes.items()
        if key.startswith("granularity_projection/")
    }
    recorded_projection_extension = dict(
        candidate_manifest.get("extensions", {})
    ).get("granularity_projection")
    if bool(recorded_projection) != (
        recorded_projection_extension is not None
    ):
        raise RuntimeError(
            "Granularity is only partially bound to the main "
            "manifest."
        )
    if recorded_projection_extension is not None:
        projection_manifest_relative = str(
            recorded_projection_extension["manifest"]
        )
        projection_manifest_path = (
            backend_dir / projection_manifest_relative
        )
        if (
            not projection_manifest_path.is_file()
            or sha256_file(projection_manifest_path)
            != recorded_projection_extension["manifest_sha256"]
        ):
            raise RuntimeError(
                "Granularity submanifest is missing or differs."
            )
        projection_manifest = json.loads(
            projection_manifest_path.read_text(encoding="utf-8")
        )
        expected_projection = {
            str(
                (projection_manifest_path.parent / filename)
                .relative_to(backend_dir)
            ): str(digest)
            for filename, digest in projection_manifest.get(
                "files", {}
            ).items()
        }
        if recorded_projection != expected_projection:
            raise RuntimeError(
                "Main and granularity submanifests differ."
            )
        for relative, expected_hash in expected_projection.items():
            projection_artifact = backend_dir / relative
            if (
                not projection_artifact.is_file()
                or sha256_file(projection_artifact) != expected_hash
            ):
                raise RuntimeError(
                    "Granularity artifact is missing or differs: "
                    f"{relative}"
                )
    teacher_agreement_extension = dict(
        candidate_manifest.get("extensions", {})
    ).get("teacher_dev_agreement")
    if teacher_agreement_extension is not None:
        if teacher_agreement_extension.get("scope") != (
            "result_dir"
        ):
            raise RuntimeError(
                "Teacher-agreement extension has an unknown "
                "scope."
            )
        teacher_manifest_path = RESULT_DIR / str(
            teacher_agreement_extension["manifest"]
        )
        if (
            not teacher_manifest_path.is_file()
            or sha256_file(teacher_manifest_path)
            != teacher_agreement_extension["manifest_sha256"]
        ):
            raise RuntimeError(
                "Teacher-agreement submanifest is missing or "
                "differs."
            )
        teacher_manifest = json.loads(
            teacher_manifest_path.read_text(encoding="utf-8")
        )
        if teacher_manifest.get("complete") is not True:
            raise RuntimeError(
                "Teacher-agreement submanifest is incomplete."
            )
        for filename, expected_hash in teacher_manifest.get(
            "files", {}
        ).items():
            teacher_artifact = teacher_manifest_path.parent / filename
            if (
                not teacher_artifact.is_file()
                or sha256_file(teacher_artifact) != expected_hash
            ):
                raise RuntimeError(
                    "Teacher-agreement artifact is missing or differs: "
                    f"{filename}"
                )
    files_match = True
    for relative in core_analysis_files:
        artifact = backend_dir / relative
        expected_hash = recorded_hashes.get(relative)
        if (
            expected_hash is None
            or not artifact.is_file()
            or sha256_file(artifact) != expected_hash
        ):
            files_match = False
            break
    if identity_matches and files_match:
        existing_final_manifest = candidate_manifest
        REUSE_COMPLETE_ANALYSIS = True

display(pd.DataFrame([{
    "reuse_frozen_query_bundle": REUSE_FROZEN_BUNDLE,
    "reuse_complete_backend_analysis": REUSE_COMPLETE_ANALYSIS,
    "backend": BACKEND,
    "query_bundle_sha256": bundle_sha256,
}]))


,dataset,queries,counts,sample_id_sha256
0,TopiOCQA dev,2514,"{'all': 2514, 'history_0': 205, 'history_1': 2...",f2a71cab647da97ea16471715df20afb268c92833e2341...


,reuse_frozen_query_bundle,reuse_complete_backend_analysis,backend,query_bundle_sha256
0,True,False,ance,c7358e2430302918d218f41e0d55ffb8a1991773872aad...


The final population contains all 2,514 TopiOCQA development queries, including 205 without history, 205 with one history turn, and 2,104 with at least two turns. Its sample-ID hash fixes the exact evaluation population.

Both reuse checks passed. The frozen query bundle and the selected complete backend analysis match the registered protocol and file hashes, so the verified results can be loaded without repeating selector inference, IterCQR rewriting, or retrieval.

### Generating selector history candidates

Each checkpoint must be converted into concrete history selections before retrieval can be evaluated. Comparing the pretrained, imitation-control, and BM25-treatment checkpoints isolates whether retrieval-guided post-training changes the selected context and improves retrieval.

Viterbi supplies the selector’s highest-probability history. Two FFBS draws expose alternative histories supported by the CRF distribution. These alternatives test whether sequence uncertainty contains complementary retrieval evidence that can be recovered through RRF fusion.

Per-query SHA-256 seeds make the sampled histories reproducible. Persisting the candidates ensures that BM25 and ANCE receive identical selector outputs, so differences between backends cannot result from new CRF samples. Reuse also avoids repeating MiniLM–CRF inference when the frozen candidates already exist.

In [5]:
if REUSE_FROZEN_BUNDLE:
    selector_predictions = {}
    selector_parameters = pd.DataFrame([
        {
            "system": system,
            "total_parameters": int(
                pretrained_manifest["parameters"]["total_parameters"]
            ),
            "trainable_parameters": int(
                pretrained_manifest["parameters"]["trainable_parameters"]
            ),
        }
        for system in SYSTEM_ORDER
    ])
    if not REUSE_COMPLETE_ANALYSIS:
        prediction_root = RESULT_DIR / "predictions"
        for system in SYSTEM_ORDER:
            system_dir = prediction_root / system
            manifest_path = system_dir / "manifest.json"
            viterbi_path = system_dir / "viterbi.jsonl"
            if not manifest_path.is_file() or not viterbi_path.is_file():
                raise FileNotFoundError(
                    f"Persisted Viterbi selection is missing: {system_dir}"
                )
            manifest = json.loads(
                manifest_path.read_text(encoding="utf-8")
            )
            required = {
                "system": system,
                "checkpoint_sha256": EXPECTED_CHECKPOINT_HASHES[system],
                "population_sha256": population_sha256,
            }
            for field, expected in required.items():
                if manifest.get(field) != expected:
                    raise RuntimeError(
                        f"Prediction manifest differs: {system}/{field}"
                    )
            recorded = manifest.get("files", {}).get("viterbi", {})
            if recorded.get("sha256") != sha256_file(viterbi_path):
                raise RuntimeError(f"Viterbi file hash differs: {system}")
            viterbi = read_selected_histories_jsonl(
                viterbi_path, progress=True
            )
            if set(viterbi) != set(sample_ids):
                raise RuntimeError(f"Viterbi population differs: {system}")
            selector_predictions[system] = {"viterbi": viterbi}
    display(selector_parameters)
    display(pd.DataFrame([{
        "selector_source": "frozen_artifacts",
        "selector_inference_executed": False,
    }]))
else:
    selector_tokenizer = AutoTokenizer.from_pretrained(
        SELECTOR_CONFIG.model_name,
        revision=pretrained_config["model_revision"],
        use_fast=True,
    )
    prediction_root = RESULT_DIR / "predictions"
    prediction_root.mkdir(parents=True, exist_ok=True)
    inference_rows = [
        row for row in dev_rows if int(row["history_len"]) >= 1
    ]
    no_history_ids = [
        str(row["sample_id"])
        for row in dev_rows
        if int(row["history_len"]) == 0
    ]
    selector_predictions: dict[
        str,
        dict[str, dict[str, list[dict[str, Any]]]],
    ] = {}
    selector_parameter_rows: list[dict[str, Any]] = []


    def load_or_predict_selector(system: str) -> dict[str, Any]:
        system_dir = prediction_root / system
        manifest_path = system_dir / "manifest.json"
        output_paths = {
            arm: system_dir / f"{arm}.jsonl"
            for arm in ("viterbi", "sample_1", "sample_2")
        }
        expected_manifest = {
            "schema_version": 1,
            "system": system,
            "checkpoint_sha256": EXPECTED_CHECKPOINT_HASHES[system],
            "population_sha256": population_sha256,
            "sampling_seed": CRF_SAMPLING_SEED,
            "sampling_seed_scheme": SAMPLING_SEED_SCHEME,
            "num_ffbs_samples": NUM_FFBS_SAMPLES,
        }
        if manifest_path.exists() and all(
            path.exists() for path in output_paths.values()
        ):
            manifest = json.loads(
                manifest_path.read_text(encoding="utf-8")
            )
            if all(
                manifest.get(key) == value
                for key, value in expected_manifest.items()
            ):
                loaded = {
                    arm: read_selected_histories_jsonl(
                        path,
                        progress=True,
                    )
                    for arm, path in output_paths.items()
                }
                if all(set(rows) == set(sample_ids) for rows in loaded.values()):
                    return loaded

        model = load_crf_inference_model(
            CHECKPOINTS[system],
            model_name=SELECTOR_CONFIG.model_name,
            model_revision=pretrained_config["model_revision"],
            num_labels=3,
            device=DEVICE,
        )
        selector_parameter_rows.append(
            {
                "system": system,
                **selector_parameter_counts(model),
            }
        )
        predicted = predict_crf_history_candidates(
            inference_rows,
            model=model,
            tokenizer=selector_tokenizer,
            device=DEVICE,
            config=SELECTOR_CONFIG,
            batch_size=SELECTOR_BATCH_SIZE,
            num_samples=NUM_FFBS_SAMPLES,
            sampling_seed=CRF_SAMPLING_SEED,
            sampling_seed_mode="per_query_sha256",
            progress=True,
            progress_desc=f"ROCC {system}",
        )
        for arm in predicted:
            for sample_id in no_history_ids:
                predicted[arm][sample_id] = []
            write_selected_histories_jsonl(
                output_paths[arm],
                predicted[arm],
                sample_order=sample_ids,
                progress=True,
            )
        seed_rows = [
            {
                "sample_id": sample_id,
                "candidate": candidate,
                "query_seed": derive_query_seed(
                    CRF_SAMPLING_SEED,
                    sample_id,
                    candidate,
                ),
            }
            for sample_id in sample_ids
            for candidate in ("ffbs_1", "ffbs_2")
        ]
        write_frame(system_dir / "sampling_seeds.csv", pd.DataFrame(seed_rows))
        write_json(
            manifest_path,
            {
                **expected_manifest,
                "files": {
                    arm: {
                        "path": str(path),
                        "sha256": sha256_file(path),
                    }
                    for arm, path in output_paths.items()
                },
                "complete": True,
            },
        )
        del model
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        return predicted


    for system in SYSTEM_ORDER:
        selector_predictions[system] = load_or_predict_selector(system)

    selector_parameters = pd.DataFrame(selector_parameter_rows)
    if selector_parameters.empty:
        selector_parameters = pd.DataFrame(
            [
                {
                    "system": system,
                    "total_parameters": int(
                        pretrained_manifest["parameters"]["total_parameters"]
                    ),
                    "trainable_parameters": int(
                        pretrained_manifest["parameters"]["trainable_parameters"]
                    ),
                }
                for system in SYSTEM_ORDER
            ]
        )
    display(selector_parameters)


,system,total_parameters,trainable_parameters
0,pretrained,33361170,33361170
1,imitation_control_e6,33361170,33361170
2,bm25_treatment_e6,33361170,33361170


,selector_source,selector_inference_executed
0,frozen_artifacts,False


All three systems use the same fully trainable 33.36-million-parameter MiniLM–CRF architecture. Their retrieval differences therefore arise from the learned checkpoint weights and training targets, not from model capacity.

Selector inference was not repeated because the validated frozen candidates were reused.

### Materializing the IterCQR retrieval queries

The `R` route is defined as `ROCC-selected history → IterCQR rewrite → retrieval`. Every selected history must therefore be serialized with the current query and converted into a standalone retrieval query. The `I` route applies the same IterCQR model to the budget-truncated full history, providing a controlled baseline in which only the history selection changes.

All budgets are materialized to measure when selection helps under compression. FFBS candidates are added at B64 to test whether CRF uncertainty contains complementary context beyond the Viterbi sequence. Persisting the resulting queries ensures that BM25 and ANCE evaluate exactly the same model outputs.

In [6]:
if REUSE_FROZEN_BUNDLE:
    serialization_dir = RESULT_DIR / "query_bundle/serialization"
    main_inputs_path = serialization_dir / "main_inputs.csv"
    if REUSE_COMPLETE_ANALYSIS:
        main_rewrites = pd.DataFrame()
    else:
        if not main_inputs_path.is_file():
            raise FileNotFoundError(
                f"Persisted main serialization is missing: {main_inputs_path}"
            )
        main_rewrites = pd.read_csv(main_inputs_path)
        required_columns = {
            "sample_id", "budget", "arm", "input_key", "input_length",
            "raw_input_length", "was_truncated", "rewrite_norm",
            "history_depth",
        }
        if not required_columns.issubset(main_rewrites.columns):
            raise RuntimeError(
                "main_inputs.csv is missing required columns."
            )
        expected_arms = {"I", *{f"{system}_R" for system in SYSTEM_ORDER}}
        if (
            len(main_rewrites)
            != len(sample_ids) * len(BUDGETS) * len(expected_arms)
            or set(main_rewrites["sample_id"].astype(str)) != set(sample_ids)
            or set(main_rewrites["budget"].astype(int)) != set(BUDGETS)
            or set(main_rewrites["arm"].astype(str)) != expected_arms
            or main_rewrites.duplicated(
                ["sample_id", "budget", "arm"]
            ).any()
        ):
            raise RuntimeError(
                "Persisted main serialization is incomplete."
            )
    ffbs_rewrites = pd.DataFrame()
    display(pd.DataFrame([{
        "serialization_source": "frozen_csv",
        "main_rows": len(main_rewrites),
        "serialization_executed": False,
        "rewrites_executed": 0,
    }]))
else:
    itercqr_tokenizer = load_itercqr_tokenizer(ITERCQR_MODEL_DIR)
    rewrite_pipeline = CachedIterCQRBM25Pipeline(
        config=PIPELINE_CONFIG,
        tokenizer=itercqr_tokenizer,
    )
    full_histories = {
        str(sample.sample_id): history_as_dicts(sample)
        for sample in dev_samples
    }
    main_histories: dict[str, Mapping[str, Sequence[dict[str, Any]]]] = {
        "I": full_histories,
    }
    for system in SYSTEM_ORDER:
        main_histories[f"{system}_R"] = selector_predictions[system][
            "viterbi"
        ]

    main_serialized, main_serialization_summary = (
        materialize_history_arm_inputs(
            samples=dev_samples,
            histories_by_arm=main_histories,
            arm_order=tuple(main_histories),
            tokenizer=itercqr_tokenizer,
            budgets=BUDGETS,
            pipeline=rewrite_pipeline,
            progress=True,
            progress_desc="serialize I/R full dev",
        )
    )
    main_rewrite_result = rewrite_pipeline.materialize_rewrites(
        main_serialized,
        progress_prefix="NB07a I/R",
    )
    main_rewrites = main_rewrite_result.serialized_with_rewrite
    main_rewrites["history_depth"] = main_rewrites["sample_id"].map(
        depth_by_sample
    )
    main_rewrites["t5_output_tokens"] = main_rewrites["rewrite"].map(
        lambda text: len(
            itercqr_tokenizer.encode(
                str(text),
                add_special_tokens=True,
            )
        )
    )

    ffbs_histories: dict[str, Mapping[str, Sequence[dict[str, Any]]]] = {}
    for system in SYSTEM_ORDER:
        ffbs_histories[f"{system}_ffbs_1"] = (
            selector_predictions[system]["sample_1"]
        )
        ffbs_histories[f"{system}_ffbs_2"] = (
            selector_predictions[system]["sample_2"]
        )
    ffbs_serialized, ffbs_serialization_summary = (
        materialize_history_arm_inputs(
            samples=dev_samples,
            histories_by_arm=ffbs_histories,
            arm_order=tuple(ffbs_histories),
            tokenizer=itercqr_tokenizer,
            budgets=(PRIMARY_BUDGET,),
            pipeline=rewrite_pipeline,
            progress=True,
            progress_desc="serialize FFBS B64",
        )
    )
    ffbs_rewrite_result = rewrite_pipeline.materialize_rewrites(
        ffbs_serialized,
        progress_prefix="NB07a FFBS",
    )
    ffbs_rewrites = ffbs_rewrite_result.serialized_with_rewrite
    ffbs_rewrites["history_depth"] = ffbs_rewrites["sample_id"].map(
        depth_by_sample
    )
    ffbs_rewrites["t5_output_tokens"] = ffbs_rewrites["rewrite"].map(
        lambda text: len(
            itercqr_tokenizer.encode(
                str(text),
                add_special_tokens=True,
            )
        )
    )

    serialization_dir = RESULT_DIR / "query_bundle/serialization"
    write_frame(serialization_dir / "main_inputs.csv", main_rewrites)
    write_frame(serialization_dir / "ffbs_inputs.csv", ffbs_rewrites)
    write_frame(
        serialization_dir / "summary.csv",
        pd.concat(
            [main_serialization_summary, ffbs_serialization_summary],
            ignore_index=True,
        ),
    )
    display(
        pd.DataFrame(
            [
                {
                    "main_rows": len(main_rewrites),
                    "main_unique_inputs": main_rewrite_result.unique_inputs,
                    "main_rewrite_cache_misses": (
                        main_rewrite_result.rewrite_cache_misses
                    ),
                    "ffbs_rows": len(ffbs_rewrites),
                    "ffbs_rewrite_cache_misses": (
                        ffbs_rewrite_result.rewrite_cache_misses
                    ),
                }
            ]
        )
    )


,serialization_source,main_rows,serialization_executed,rewrites_executed
0,frozen_csv,40224,False,0


The complete frozen query bundle and selected backend analysis have already passed validation, so the serialized inputs are not loaded into memory and IterCQR is not executed again. `main_rows = 0` therefore describes the current in-memory table, while the frozen bundle still contains all 143,298 query rows. Reuse preserves the exact evaluated queries and avoids unnecessary serialization and rewriting.

### Building the retrieval-query registry

When no frozen bundle exists, every system output is converted into a common retrieval-query format. For each selector checkpoint and budget, `I` contains the IterCQR rewrite of the budget-truncated full history, `R` contains the IterCQR rewrite of the Viterbi-selected history, and `D` uses the ROCC-selected context directly without T5. Queries without history give $I_B$ and $R_B$ the same rewriter input. The direct route $D_B$ does not rewrite this input, so its search text and ranking can differ.

At B64, a separate CRF registry stores the Viterbi rewrite and both FFBS rewrites for each checkpoint. Every row records the sample, system, budget, view, query text, normalized query, token counts, T5 requirements, and a deterministic query ID.

This registry gives BM25 and ANCE the same fixed query strings and enables paired route, decoder, fusion, and cost comparisons. When the verified bundle already exists, query reconstruction is skipped and the frozen registry is reused unchanged.

In [7]:
if REUSE_FROZEN_BUNDLE:
    rocc_query_frame = pd.DataFrame()
    crf_query_frame = pd.DataFrame()
    display(pd.DataFrame([{
        "query_source": "frozen_query_bundle",
        "direct_queries_materialized": 0,
        "crf_queries_reconstructed": 0,
    }]))
else:
    rocc_query_rows: list[dict[str, Any]] = []
    crf_query_rows: list[dict[str, Any]] = []


    def make_query_row(
        *,
        family: str,
        system: str,
        sample_id: str,
        budget: int,
        view: str,
        query: str,
        candidate_index: int,
        input_key: str | None,
        t5_input_tokens: int,
        t5_output_tokens: int,
        t5_encoder_required: bool,
        t5_decoder_required: bool,
    ) -> dict[str, Any]:
        normalized = normalize_rewrite(query)
        row = {
            "family": family,
            "system": system,
            "sample_id": str(sample_id),
            "history_depth": int(depth_by_sample[str(sample_id)]),
            "budget": int(budget),
            "view": str(view),
            "candidate_index": int(candidate_index),
            "query": str(query),
            "query_norm": normalized,
            "t5_input_key": input_key,
            "t5_input_tokens": int(t5_input_tokens),
            "t5_output_tokens": int(t5_output_tokens),
            "t5_encoder_required": bool(t5_encoder_required),
            "t5_decoder_required": bool(t5_decoder_required),
        }
        row["query_id"] = query_id_for(row)
        return row


    for system in SYSTEM_ORDER:
        for budget in tqdm(
            BUDGETS,
            desc=f"materialize D {system}",
            unit="budget",
        ):
            i_rows = main_rewrites.loc[
                main_rewrites["budget"].eq(budget)
                & main_rewrites["arm"].eq("I")
            ].sort_values("sample_id", kind="mergesort")
            r_arm = f"{system}_R"
            r_rows = main_rewrites.loc[
                main_rewrites["budget"].eq(budget)
                & main_rewrites["arm"].eq(r_arm)
            ].sort_values("sample_id", kind="mergesort")
            d_rows = materialize_selector_direct_queries(
                pipeline_results=main_rewrites,
                selector_arm=r_arm,
                budget=budget,
                config=EXPERIMENT_CONFIG,
                device=str(DEVICE),
                progress=True,
            ).set_index("sample_id")
            i_lookup = i_rows.set_index("sample_id")
            for source in r_rows.itertuples(index=False):
                sample_id = str(source.sample_id)
                i_source = i_lookup.loc[sample_id]
                no_history = int(depth_by_sample[sample_id]) == 0
                i_query = str(i_source["rewrite"])
                r_query = i_query if no_history else str(source.rewrite)
                d_query = (
                    str(sample_by_id[sample_id].current_query)
                    if no_history
                    else str(d_rows.loc[sample_id, "direct_query"])
                )
                rocc_query_rows.append(
                    make_query_row(
                        family="rocc",
                        system=system,
                        sample_id=sample_id,
                        budget=budget,
                        view="I",
                        query=i_query,
                        candidate_index=0,
                        input_key=str(i_source["input_key"]),
                        t5_input_tokens=int(i_source["input_length"]),
                        t5_output_tokens=int(i_source["t5_output_tokens"]),
                        t5_encoder_required=True,
                        t5_decoder_required=True,
                    )
                )
                rocc_query_rows.append(
                    make_query_row(
                        family="rocc",
                        system=system,
                        sample_id=sample_id,
                        budget=budget,
                        view="R",
                        query=r_query,
                        candidate_index=0,
                        input_key=(
                            str(i_source["input_key"])
                            if no_history
                            else str(source.input_key)
                        ),
                        t5_input_tokens=(
                            int(i_source["input_length"])
                            if no_history
                            else int(source.input_length)
                        ),
                        t5_output_tokens=(
                            int(i_source["t5_output_tokens"])
                            if no_history
                            else int(source.t5_output_tokens)
                        ),
                        t5_encoder_required=True,
                        t5_decoder_required=True,
                    )
                )
                rocc_query_rows.append(
                    make_query_row(
                        family="rocc",
                        system=system,
                        sample_id=sample_id,
                        budget=budget,
                        view="D",
                        query=d_query,
                        candidate_index=0,
                        input_key=None,
                        t5_input_tokens=0,
                        t5_output_tokens=0,
                        t5_encoder_required=False,
                        t5_decoder_required=False,
                    )
                )

    for system in SYSTEM_ORDER:
        viterbi_rows = main_rewrites.loc[
            main_rewrites["budget"].eq(PRIMARY_BUDGET)
            & main_rewrites["arm"].eq(f"{system}_R")
        ].sort_values("sample_id", kind="mergesort")
        ffbs_by_arm = {
            arm: ffbs_rewrites.loc[
                ffbs_rewrites["arm"].eq(f"{system}_{arm}")
            ].set_index("sample_id")
            for arm in ("ffbs_1", "ffbs_2")
        }
        for source in viterbi_rows.itertuples(index=False):
            sample_id = str(source.sample_id)
            crf_query_rows.append(
                make_query_row(
                    family="crf",
                    system=system,
                    sample_id=sample_id,
                    budget=PRIMARY_BUDGET,
                    view="viterbi",
                    query=str(source.rewrite),
                    candidate_index=0,
                    input_key=str(source.input_key),
                    t5_input_tokens=int(source.input_length),
                    t5_output_tokens=int(source.t5_output_tokens),
                    t5_encoder_required=True,
                    t5_decoder_required=True,
                )
            )
            for candidate_index, arm in enumerate(
                ("ffbs_1", "ffbs_2"),
                start=1,
            ):
                candidate = ffbs_by_arm[arm].loc[sample_id]
                crf_query_rows.append(
                    make_query_row(
                        family="crf",
                        system=system,
                        sample_id=sample_id,
                        budget=PRIMARY_BUDGET,
                        view=arm,
                        query=str(candidate["rewrite"]),
                        candidate_index=candidate_index,
                        input_key=str(candidate["input_key"]),
                        t5_input_tokens=int(candidate["input_length"]),
                        t5_output_tokens=int(candidate["t5_output_tokens"]),
                        t5_encoder_required=True,
                        t5_decoder_required=True,
                    )
                )

    rocc_query_frame = pd.DataFrame(rocc_query_rows)
    crf_query_frame = pd.DataFrame(crf_query_rows)
    display(
        rocc_query_frame.groupby(
            ["system", "budget", "view"],
            observed=True,
        )["sample_id"]
        .nunique()
        .rename("queries")
        .reset_index()
    )


,query_source,direct_queries_materialized,crf_queries_reconstructed
0,frozen_query_bundle,0,0


The verified frozen bundle already contains all direct and CRF queries. Both zeros report work performed in the current run, not the number of queries stored in the bundle.

### Adding the IterCQR sampling baseline

IterCQR can generate several plausible rewrites from the same history. A greedy rewrite and two sampled rewrites are produced at every budget. The thesis reports the matched B64 comparison $G+2S$ against $IRD_{64}$ to test whether the gain comes from different retrieval views or from T5 sampling alone. The other budgets remain supplemental notebook results.

The greedy output must match the previously cached IterCQR rewrite. The two samples use the fixed seed, temperature, top-k, and top-p configuration. Queries without history reuse the greedy output because no alternative history context exists.

The duplicate summary measures how often the three requested rewrites collapse to one or two unique queries. This establishes how much genuine query diversity IterCQR sampling provides.

Finally, the IterCQR candidates are combined with the `I`, `R`, `D`, Viterbi, and FFBS queries into one uniquely indexed bundle. The bundle is generated once during the local BM25 run and then reused unchanged by ANCE, ensuring that both backends evaluate identical deterministic and sampled queries.

In [8]:
if REUSE_COMPLETE_ANALYSIS:
    query_artifacts = pd.DataFrame()
    iter_candidate_frame = pd.DataFrame()
    duplicate_path = (
        RESULT_DIR / "query_bundle/itercqr_duplicate_summary.csv"
    )
    iter_duplicate_summary = (
        pd.read_csv(duplicate_path)
        if duplicate_path.is_file()
        else pd.DataFrame()
    )
    t5_parameter_count = {
        "total_parameters": 222_903_552,
        "trainable_parameters": 222_903_552,
    }
    display(iter_duplicate_summary.round(4))
    display(pd.DataFrame([{
        "backend": BACKEND,
        "query_source": "complete_frozen_analysis",
        "query_rows": int(frozen_bundle_manifest["rows"]),
        "query_bundle_sha256": bundle_sha256,
        "rewrites_executed": 0,
    }]))
else:
    bundle_exists = QUERY_BUNDLE_PATH.is_file()
    manifest_exists = QUERY_BUNDLE_MANIFEST_PATH.is_file()

    if bundle_exists != manifest_exists:
        raise RuntimeError(
            "Query bundle and manifest are only partially present."
        )

    reuse_frozen_bundle = bundle_exists and manifest_exists

    if BACKEND == "ance" and not reuse_frozen_bundle:
        raise FileNotFoundError(
            "ANCE requires the locally frozen query bundle."
        )

    if reuse_frozen_bundle:
        if not QUERY_BUNDLE_PATH.exists():
            raise FileNotFoundError(
                "The locally frozen query bundle is missing: "
                f"{QUERY_BUNDLE_PATH}"
            )
        if not QUERY_BUNDLE_MANIFEST_PATH.exists():
            raise FileNotFoundError(
                "The query-bundle manifest is missing: "
                f"{QUERY_BUNDLE_MANIFEST_PATH}"
            )

        frozen_manifest = json.loads(
            QUERY_BUNDLE_MANIFEST_PATH.read_text(
                encoding="utf-8"
            )
        )
        observed_bundle_sha256 = sha256_file(
            QUERY_BUNDLE_PATH
        )

        if (
            str(frozen_manifest["query_bundle_sha256"])
            != observed_bundle_sha256
        ):
            raise RuntimeError(
                "The query-bundle hash does not match."
            )
        if (
            str(frozen_manifest["population_sha256"])
            != population_sha256
        ):
            raise RuntimeError(
                "The query bundle belongs to a different population."
            )

        with QUERY_BUNDLE_PATH.open(
            "r",
            encoding="utf-8",
        ) as handle:
            query_artifacts = pd.DataFrame(
                [
                    json.loads(line)
                    for line in handle
                    if line.strip()
                ]
            )

        if query_artifacts["query_id"].duplicated().any():
            raise RuntimeError(
                "Duplicate query_id values in the query bundle."
            )

        iter_candidate_frame = query_artifacts.loc[
            query_artifacts["family"].eq("itercqr")
        ].copy()

        eligible_iter = iter_candidate_frame.loc[
            iter_candidate_frame["history_depth"].ge(1)
        ]
        duplicate_rows = []
        for budget, group in eligible_iter.groupby(
            "budget",
            observed=True,
        ):
            unique_per_query = group.groupby(
                "sample_id"
            )["query_norm"].nunique()
            requested = len(group)
            unique = int(unique_per_query.sum())
            duplicate_rows.append(
                {
                    "budget": int(budget),
                    "eligible_queries": int(
                        unique_per_query.size
                    ),
                    "requested_rewrites": int(requested),
                    "unique_rewrites": unique,
                    "duplicate_rate": (
                        (requested - unique) / requested
                    ),
                    "queries_with_duplicate_share": float(
                        unique_per_query.lt(3).mean()
                    ),
                    "one_unique": int(
                        unique_per_query.eq(1).sum()
                    ),
                    "two_unique": int(
                        unique_per_query.eq(2).sum()
                    ),
                    "three_unique": int(
                        unique_per_query.eq(3).sum()
                    ),
                }
            )
        iter_duplicate_summary = pd.DataFrame(
            duplicate_rows
        )

        # Static model size, so no model loading is required.
        t5_parameter_count = {
            "total_parameters": 222_903_552,
            "trainable_parameters": 222_903_552,
        }

        if "rewrite_pipeline" in globals():
            rewrite_pipeline.close()
            del rewrite_pipeline
        if "iter_rewriter" in globals():
            del iter_rewriter

        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

        display(iter_duplicate_summary.round(4))
        display(
            pd.DataFrame(
                [
                    {
                        "backend": BACKEND,
                        "query_source": (
                            "frozen_local_bundle"
                        ),
                        "query_rows": len(query_artifacts),
                        "query_bundle_sha256": (
                            observed_bundle_sha256
                        ),
                        "rewrites_executed": 0,
                    }
                ]
            )
        )

    else:
        # Generate only during the first local BM25 run.
        iter_candidate_rows: list[
            dict[str, Any]
        ] = []
        iter_rewriter = load_itercqr_rewriter(
            model_dir=ITERCQR_MODEL_DIR,
            device=DEVICE,
        )

        for budget in BUDGETS:
            budget_rows = main_rewrites.loc[
                main_rewrites["budget"].eq(budget)
                & main_rewrites["arm"].eq("I")
            ].sort_values(
                "sample_id",
                kind="mergesort",
            )
            generation_rows = budget_rows.loc[
                budget_rows["history_depth"].ge(1)
            ]
            no_history_rows = budget_rows.loc[
                budget_rows["history_depth"].eq(0)
            ]

            for source in no_history_rows.itertuples(
                index=False
            ):
                query = str(source.rewrite)
                output_tokens = len(
                    itercqr_tokenizer.encode(
                        query,
                        add_special_tokens=True,
                    )
                )
                for candidate_index, view in enumerate(
                    ("greedy", "sample_1", "sample_2")
                ):
                    is_greedy = view == "greedy"
                    iter_candidate_rows.append(
                        make_query_row(
                            family="itercqr",
                            system="itercqr",
                            sample_id=str(source.sample_id),
                            budget=budget,
                            view=view,
                            candidate_index=candidate_index,
                            query=query,
                            input_key=str(source.input_key),
                            t5_input_tokens=(
                                int(source.input_length)
                                if is_greedy
                                else 0
                            ),
                            t5_output_tokens=(
                                output_tokens
                                if is_greedy
                                else 0
                            ),
                            t5_encoder_required=is_greedy,
                            t5_decoder_required=is_greedy,
                        )
                    )

            starts = range(
                0,
                len(generation_rows),
                REWRITE_BATCH_SIZE,
            )
            for batch_index, start in tqdm(
                enumerate(starts),
                total=math.ceil(
                    len(generation_rows)
                    / REWRITE_BATCH_SIZE
                ),
                desc=f"IterCQR G+2S B{budget}",
                unit="batch",
            ):
                batch_rows = generation_rows.iloc[
                    start : start + REWRITE_BATCH_SIZE
                ]
                generated = (
                    iter_rewriter
                    .rewrite_greedy_and_samples_batch(
                        rewrite_pipeline.make_rewriter_batch(
                            batch_rows[
                                "input_key"
                            ].astype(str).tolist()
                        ),
                        sample_ids=batch_rows[
                            "sample_id"
                        ].astype(str).tolist(),
                        num_samples=NUM_ITER_SAMPLES,
                        base_sampling_seed=(
                            ITER_SAMPLING_SEED
                        ),
                        sampling_temperature=(
                            ITER_SAMPLING_TEMPERATURE
                        ),
                        sampling_top_k=(
                            ITER_SAMPLING_TOP_K
                        ),
                        sampling_top_p=(
                            ITER_SAMPLING_TOP_P
                        ),
                    )
                )

                for row_index, source in enumerate(
                    batch_rows.itertuples(index=False)
                ):
                    sample_id = str(source.sample_id)
                    greedy = str(
                        generated["greedy"][row_index]
                    )
                    if (
                        normalize_rewrite(greedy)
                        != str(source.rewrite_norm)
                    ):
                        raise RuntimeError(
                            f"Greedy output differs for "
                            f"{sample_id}, B{budget}."
                        )

                    candidates = {
                        "greedy": greedy,
                        "sample_1": str(
                            generated["sample_1"][row_index]
                        ),
                        "sample_2": str(
                            generated["sample_2"][row_index]
                        ),
                    }
                    no_history = (
                        int(depth_by_sample[sample_id]) == 0
                    )
                    if no_history:
                        candidates["sample_1"] = greedy
                        candidates["sample_2"] = greedy

                    for candidate_index, view in enumerate(
                        (
                            "greedy",
                            "sample_1",
                            "sample_2",
                        )
                    ):
                        decoder_required = not (
                            no_history and view != "greedy"
                        )
                        query = candidates[view]
                        iter_candidate_rows.append(
                            make_query_row(
                                family="itercqr",
                                system="itercqr",
                                sample_id=sample_id,
                                budget=budget,
                                view=view,
                                query=query,
                                candidate_index=(
                                    candidate_index
                                ),
                                input_key=str(
                                    source.input_key
                                ),
                                t5_input_tokens=(
                                    int(source.input_length)
                                    if view == "greedy"
                                    else 0
                                ),
                                t5_output_tokens=(
                                    len(
                                        itercqr_tokenizer.encode(
                                            query,
                                            add_special_tokens=True,
                                        )
                                    )
                                    if decoder_required
                                    else 0
                                ),
                                t5_encoder_required=(
                                    view == "greedy"
                                ),
                                t5_decoder_required=(
                                    decoder_required
                                ),
                            )
                        )

        iter_candidate_frame = pd.DataFrame(
            iter_candidate_rows
        )
        eligible_iter = iter_candidate_frame.loc[
            iter_candidate_frame["history_depth"].ge(1)
        ]

        duplicate_rows = []
        for budget, group in eligible_iter.groupby(
            "budget",
            observed=True,
        ):
            unique_per_query = group.groupby(
                "sample_id"
            )["query_norm"].nunique()
            requested = len(group)
            unique = int(unique_per_query.sum())
            duplicate_rows.append(
                {
                    "budget": int(budget),
                    "eligible_queries": int(
                        unique_per_query.size
                    ),
                    "requested_rewrites": int(requested),
                    "unique_rewrites": unique,
                    "duplicate_rate": (
                        (requested - unique) / requested
                    ),
                    "queries_with_duplicate_share": float(
                        unique_per_query.lt(3).mean()
                    ),
                    "one_unique": int(
                        unique_per_query.eq(1).sum()
                    ),
                    "two_unique": int(
                        unique_per_query.eq(2).sum()
                    ),
                    "three_unique": int(
                        unique_per_query.eq(3).sum()
                    ),
                }
            )
        iter_duplicate_summary = pd.DataFrame(
            duplicate_rows
        )

        t5_parameter_count = selector_parameter_counts(
            iter_rewriter.model
        )

        write_frame(
            RESULT_DIR
            / "query_bundle/itercqr_duplicate_summary.csv",
            iter_duplicate_summary,
        )
        display(iter_duplicate_summary.round(4))

        query_artifacts = pd.concat(
            [
                rocc_query_frame,
                crf_query_frame,
                iter_candidate_frame,
            ],
            ignore_index=True,
        )
        if query_artifacts["query_id"].duplicated().any():
            raise RuntimeError(
                "Duplicate query_id values in the query bundle."
            )

        del iter_rewriter
        rewrite_pipeline.close()
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()


,budget,eligible_queries,requested_rewrites,unique_rewrites,duplicate_rate,queries_with_duplicate_share,one_unique,two_unique,three_unique
0,64,2309,6927,5928,0.1442,0.3551,179,641,1489
1,128,2309,6927,5732,0.1725,0.4270,209,777,1323
2,256,2309,6927,5754,0.1693,0.4214,200,773,1336
3,512,2309,6927,5735,0.1721,0.4262,208,776,1325


,backend,query_source,query_rows,query_bundle_sha256,rewrites_executed
0,ance,frozen_local_bundle,143298,c7358e2430302918d218f41e0d55ffb8a1991773872aad...,0


Across the 2,309 queries with history, IterCQR sampling produces substantial but incomplete query diversity. At B64, 64.5% yield three unique rewrites and the duplicate rate is 14.4%, increasing to approximately 17% at larger budgets. This matters because RRF benefits from distinct candidates. The verified bundle freezes all 143,298 query variants so BM25 and ANCE evaluate and fuse exactly the same inputs.

### Freezing and binding the query bundle

All `I`, `R`, `D`, Viterbi, FFBS, and IterCQR queries are projected onto one fixed schema, sorted deterministically, and serialized as JSONL. A SHA-256 hash identifies the complete payload. The local BM25 run creates this bundle, while ANCE requires and reuses the same frozen file.

When query artifacts are available, their deterministic serialization is compared with the frozen bundle. A BM25 mismatch stops execution. ANCE continues to use the frozen local bundle so retrieval quality remains based on the same query set. Duplicate query IDs are rejected. During complete-analysis reuse, no reconstruction occurs and the previously validated bundle remains read-only.

The manifest binds the bundle to the population, selector checkpoints, IterCQR model, serializer, budgets, and sampling configuration. This ensures that BM25 and ANCE evaluate the same generated queries and that every result can be traced to the protocol that produced them. `selection_performed=False` records that the development split was used for evaluation without model or route selection.

In [9]:
if REUSE_COMPLETE_ANALYSIS:
    QUERY_BUNDLE_COLUMNS = (
        "family", "system", "sample_id", "history_depth", "budget",
        "view", "candidate_index", "query_id", "query", "query_norm",
        "t5_input_key", "t5_input_tokens", "t5_output_tokens",
        "t5_encoder_required", "t5_decoder_required",
    )
    reconstructed_bundle_sha256 = str(bundle_sha256)
    reconstruction_matches_bundle = True
    query_bundle_manifest = dict(frozen_bundle_manifest)
    display(pd.DataFrame([{
        "query_rows": int(query_bundle_manifest["rows"]),
        "unique_query_ids": int(query_bundle_manifest["rows"]),
        "query_bundle_sha256": bundle_sha256,
        "reconstruction_matches": True,
        "bundle_read_only": True,
    }]))
else:
    QUERY_BUNDLE_COLUMNS = (
        "family",
        "system",
        "sample_id",
        "history_depth",
        "budget",
        "view",
        "candidate_index",
        "query_id",
        "query",
        "query_norm",
        "t5_input_key",
        "t5_input_tokens",
        "t5_output_tokens",
        "t5_encoder_required",
        "t5_decoder_required",
    )
    reconstructed_query_artifacts = query_artifacts.loc[
        :, QUERY_BUNDLE_COLUMNS
    ].sort_values(
        [
            "family",
            "system",
            "budget",
            "sample_id",
            "view",
            "candidate_index",
        ],
        kind="mergesort",
    ).reset_index(drop=True)
    reconstructed_payload = "".join(
        canonical_json(row) + "\n"
        for row in reconstructed_query_artifacts.to_dict(
            orient="records"
        )
    )
    reconstructed_bundle_sha256 = sha256_text(
        reconstructed_payload
    )

    if QUERY_BUNDLE_PATH.exists():
        bundle_sha256 = sha256_file(QUERY_BUNDLE_PATH)
        with QUERY_BUNDLE_PATH.open("r", encoding="utf-8") as handle:
            frozen_rows = [json.loads(line) for line in handle if line.strip()]
        query_artifacts = pd.DataFrame(frozen_rows).loc[
            :, QUERY_BUNDLE_COLUMNS
        ]
        reconstruction_matches_bundle = (
            reconstructed_bundle_sha256 == bundle_sha256
        )
        if BACKEND == "bm25" and not reconstruction_matches_bundle:
            raise RuntimeError(
                "The local reconstruction differs from the frozen "
                "query bundle."
            )
        if BACKEND == "ance" and not reconstruction_matches_bundle:
            print(
                "Note: The Colab reconstruction differs numerically. "
                "ANCE quality still uses the unchanged locally frozen "
                "query bundle."
            )
    else:
        if BACKEND == "ance":
            raise FileNotFoundError(
                "ANCE requires the previously frozen local query bundle: "
                f"{QUERY_BUNDLE_PATH}"
            )
        QUERY_BUNDLE_PATH.parent.mkdir(parents=True, exist_ok=True)
        temporary = QUERY_BUNDLE_PATH.with_name(
            f".{QUERY_BUNDLE_PATH.name}.tmp"
        )
        temporary.write_text(
            reconstructed_payload,
            encoding="utf-8",
        )
        temporary.replace(QUERY_BUNDLE_PATH)
        query_artifacts = reconstructed_query_artifacts
        bundle_sha256 = sha256_file(QUERY_BUNDLE_PATH)
        reconstruction_matches_bundle = True

    if query_artifacts["query_id"].duplicated().any():
        raise RuntimeError("The frozen query bundle contains duplicates.")
    write_json(
        RESULT_DIR / f"query_bundle/reconstruction_{BACKEND}.json",
        {
            "backend": BACKEND,
            "reconstructed_query_bundle_sha256": (
                reconstructed_bundle_sha256
            ),
            "frozen_query_bundle_sha256": bundle_sha256,
            "matches": reconstruction_matches_bundle,
            "quality_uses_frozen_bundle": True,
        },
    )

    query_bundle_manifest = {
        "schema_version": 1,
        "protocol": "nb07a_topiocqa_final_v2",
        "d_no_history_query_rule": D_NO_HISTORY_QUERY_RULE,
        "population_sha256": population_sha256,
        "query_bundle_sha256": bundle_sha256,
        "rows": len(query_artifacts),
        "budgets": list(BUDGETS),
        "checkpoint_sha256": EXPECTED_CHECKPOINT_HASHES,
        "itercqr_model_sha256": sha256_file(
            ITERCQR_MODEL_DIR / "pytorch_model.bin"
        ),
        "serializer_sha256": sha256_text(
            inspect.getsource(serialize_itercqr_input)
        ),
        "sampling_seed_scheme": SAMPLING_SEED_SCHEME,
        "iter_sampling_seed": ITER_SAMPLING_SEED,
        "iter_sampling_temperature": ITER_SAMPLING_TEMPERATURE,
        "iter_sampling_top_k": ITER_SAMPLING_TOP_K,
        "iter_sampling_top_p": ITER_SAMPLING_TOP_P,
        "crf_sampling_seed": CRF_SAMPLING_SEED,
        "candidate_names": [
            "itercqr_sample_1",
            "itercqr_sample_2",
            "ffbs_1",
            "ffbs_2",
        ],
        "complete": True,
        "selection_performed": False,
    }
    if QUERY_BUNDLE_MANIFEST_PATH.exists():
        existing_manifest = json.loads(
            QUERY_BUNDLE_MANIFEST_PATH.read_text(encoding="utf-8")
        )
        if existing_manifest != query_bundle_manifest:
            raise RuntimeError("Query-bundle manifest differs.")
    else:
        if BACKEND == "ance":
            raise FileNotFoundError(
                "ANCE requires the local query-bundle manifest."
            )
        write_json(QUERY_BUNDLE_MANIFEST_PATH, query_bundle_manifest)

    display(
        pd.DataFrame(
            [
                {
                    "query_rows": len(query_artifacts),
                    "unique_query_ids": query_artifacts["query_id"].nunique(),
                    "unique_query_texts": query_artifacts[
                        "query_norm"
                    ].nunique(),
                    "query_bundle_sha256": bundle_sha256,
                    "reconstruction_matches": reconstruction_matches_bundle,
                }
            ]
        )
    )


,query_rows,unique_query_ids,unique_query_texts,query_bundle_sha256,reconstruction_matches
0,143298,143298,37788,c7358e2430302918d218f41e0d55ffb8a1991773872aad...,True


The frozen bundle contains 143,298 query records with unique IDs and is identified by its SHA-256 hash. It is reused read-only. `reconstruction_matches=True` refers to the previously validated bundle identity because no reconstruction was executed in this run.

### Preparing and validating the retrieval backend

The selected backend receives the frozen query bundle and returns up to 1,000 passages for each query. When complete analysis artifacts already exist, their retrieval manifest is checked against the backend, query-bundle hash, and retrieval depth. The stored backend identity and runtime metadata are then reused without loading a retriever.

For an incomplete analysis, existing rankings are reused only when the ranking file exists, its SHA-256 hash matches the manifest, and the manifest refers to the current backend and query bundle. Otherwise, BM25 validates and loads the TopiOCQA Lucene index with \(k_1=0.9\) and \(b=0.4\). ANCE inspects the prepared local or Drive-linked FAISS shards, checks GPU capacity, and loads the float32 dense index together with its query encoder. Missing index shards must be restored or built before this step; no private Azure download is attempted.

The backend identity, machine environment, parameter count, and reuse status are written to `runtime.json`. This makes every ranking traceable to the exact retriever configuration. Evaluating the same frozen queries with BM25 and ANCE tests whether the observed effects generalize across sparse and dense retrieval, while validated ranking reuse avoids repeating the expensive search over all queries and passages.

In [10]:
if REUSE_COMPLETE_ANALYSIS:
    backend_dir = RESULT_DIR / "backends" / BACKEND
    rankings_path = backend_dir / "rankings.jsonl.gz"
    retrieval_manifest_path = backend_dir / "retrieval_manifest.json"
    runtime_path = backend_dir / "runtime.json"
    if not retrieval_manifest_path.is_file():
        raise FileNotFoundError(retrieval_manifest_path)
    existing_retrieval_manifest = json.loads(
        retrieval_manifest_path.read_text(encoding="utf-8")
    )
    if (
        existing_retrieval_manifest.get("query_bundle_sha256")
        != bundle_sha256
        or existing_retrieval_manifest.get("backend") != BACKEND
        or existing_retrieval_manifest.get("d_no_history_query_rule")
        != D_NO_HISTORY_QUERY_RULE
        or int(existing_retrieval_manifest.get("top_k", 0))
        != RETRIEVAL_TOP_K
    ):
        raise RuntimeError("Persisted retrieval manifest differs.")
    existing_backend_runtime = (
        json.loads(runtime_path.read_text(encoding="utf-8"))
        if runtime_path.is_file() else None
    )
    reuse_rankings = True
    retriever = None
    backend_identity = dict(existing_final_manifest["backend_identity"])
    backend_parameter_count = (
        existing_backend_runtime.get("backend_parameter_count")
        if existing_backend_runtime is not None else None
    )
    backend_runtime = existing_backend_runtime
    display(pd.DataFrame([{
        **backend_identity,
        "backend_results_read_only": True,
    }]))
else:
    backend_dir = RESULT_DIR / "backends" / BACKEND
    backend_dir.mkdir(parents=True, exist_ok=True)
    rankings_path = backend_dir / "rankings.jsonl.gz"
    retrieval_manifest_path = backend_dir / "retrieval_manifest.json"
    runtime_path = backend_dir / "runtime.json"
    existing_backend_runtime = (
        json.loads(runtime_path.read_text(encoding="utf-8"))
        if runtime_path.exists()
        else None
    )
    existing_retrieval_manifest = (
        json.loads(retrieval_manifest_path.read_text(encoding="utf-8"))
        if retrieval_manifest_path.exists()
        else None
    )
    reuse_rankings = bool(
        rankings_path.exists()
        and existing_retrieval_manifest is not None
        and existing_retrieval_manifest.get("query_bundle_sha256")
        == bundle_sha256
        and existing_retrieval_manifest.get("backend") == BACKEND
        and existing_retrieval_manifest.get("d_no_history_query_rule")
        == D_NO_HISTORY_QUERY_RULE
        and int(existing_retrieval_manifest.get("top_k", 0))
        == RETRIEVAL_TOP_K
        and existing_retrieval_manifest.get("rankings_sha256")
        == sha256_file(rankings_path)
    )

    retriever = None
    backend_identity: dict[str, Any]
    backend_parameter_count: int | None = None
    if reuse_rankings:
        backend_identity = dict(existing_retrieval_manifest["backend_identity"])
        if existing_backend_runtime is not None:
            backend_parameter_count = existing_backend_runtime.get(
                "backend_parameter_count"
            )
        print("Loading rankings from the frozen backend artifact.")
    elif BACKEND == "bm25":
        bm25_status = inspect_bm25_lucene_index(
            BM25_INDEX_DIR,
            source_passages=TOPIOCQA_CORPUS_PASSAGES,
        )
        if not bm25_status.loadable:
            raise RuntimeError(
                f"TopiOCQA BM25 index cannot be loaded: {bm25_status}"
            )
        retriever = load_bm25_retriever(
            index_dir=BM25_INDEX_DIR,
            k1=0.9,
            b=0.4,
            retrieval_workers=PIPELINE_CONFIG.retrieval_workers,
        )
        backend_identity = {
            "backend": "bm25",
            "index_dir": str(BM25_INDEX_DIR.resolve()),
            "index_size_bytes": sum(
                path.stat().st_size
                for path in BM25_INDEX_DIR.rglob("*")
                if path.is_file()
            ),
            "k1": 0.9,
            "b": 0.4,
            "retrieval_workers": PIPELINE_CONFIG.retrieval_workers,
        }
    else:
        if IN_COLAB:
            dependency_status = ensure_colab_retrieval_dependencies()
            display(pd.DataFrame([dependency_status.to_dict()]))
        # Supply the complete original-format index locally or link it from Drive.
        dense_summary = summarize_dense_faiss_shards(DENSE_INDEX_DIR)
        capacity = check_dense_gpu_capacity(
            dense_summary,
            device=str(DEVICE),
            index_dtype="float32",
            reserve_gib=4.0,
        )
        if not capacity.fits:
            raise RuntimeError(f"ANCE does not fit on the GPU: {capacity}")
        retriever = load_dense_ance_torch_retriever(
            index_dir=dense_summary.index_dir,
            device=str(DEVICE),
            index_dtype="float32",
            chunk_size=50_000,
            query_batch_size=64,
            progress=True,
            status_log=PROJECT_ROOT / "experiments/logs/ance_index_load_status.log",
        )
        query_encoder = getattr(
            getattr(retriever, "encode_queries", None),
            "__self__",
            None,
        )
        query_encoder_model = getattr(query_encoder, "model", None)
        if query_encoder_model is not None and hasattr(
            query_encoder_model,
            "parameters",
        ):
            backend_parameter_count = sum(
                parameter.numel()
                for parameter in query_encoder_model.parameters()
            )
        backend_identity = {
            "backend": "ance",
            "index_dir": str(dense_summary.index_dir),
            "shards": int(dense_summary.shard_count),
            "passages": int(dense_summary.num_passages),
            "index_dtype": "float32",
            "chunk_size": 50_000,
            "query_batch_size": 64,
        }

    backend_runtime = {
        "backend": BACKEND,
        "machine": software_manifest,
        "backend_identity": backend_identity,
        "backend_parameter_count": backend_parameter_count,
        "rankings_reused": reuse_rankings,
    }
    write_json(runtime_path, backend_runtime)
    display(pd.DataFrame([backend_identity]))


Loading rankings from the frozen backend artifact.


,backend,index_dir,shards,passages,index_dtype,chunk_size,query_batch_size
0,ance,/experiments/data/pyseri...,33,25700592,float32,50000,64


The selected backend results are reused read-only. The BM25 rankings were generated from the 19.98 GB TopiOCQA Lucene index with \(k_1=0.9\), \(b=0.4\), and eight retrieval workers. The ANCE rankings were generated from 33 float32 FAISS shards containing 25,700,592 passages with a query batch size of 64. No index loading or retrieval is repeated during a validated restore run.

### Retrieving the candidate passages

Each frozen query is converted into a ranked list of 1,000 passages. These rankings are required to calculate MRR, nDCG, recall, and RRF fusion. They are stored once so every analysis uses the same retrieval results without rerunning BM25 or ANCE.

In [11]:
if REUSE_COMPLETE_ANALYSIS:
    hits_by_query_id = {}
    display(pd.DataFrame([{
        "ranking_source": "completed_metric_artifacts",
        "rankings_loaded_into_memory": False,
        "retrieval_executed": False,
    }]))
else:
    def write_rankings(
        path: Path,
        hits_by_query_id: Mapping[str, Sequence[Mapping[str, Any]]],
    ) -> None:
        temporary = path.with_name(f".{path.name}.tmp")
        with gzip.open(temporary, "wt", encoding="utf-8") as handle:
            for query_id in tqdm(
                sorted(hits_by_query_id),
                desc=f"write {path.name}",
                unit="query",
            ):
                handle.write(
                    canonical_json(
                        {
                            "query_id": query_id,
                            "docids": [
                                str(hit["docid"])
                                for hit in hits_by_query_id[query_id][
                                    :RETRIEVAL_TOP_K
                                ]
                            ],
                        }
                    )
                    + "\n"
                )
        temporary.replace(path)


    def read_rankings(
        path: Path,
    ) -> dict[str, list[dict[str, Any]]]:
        rows: dict[str, list[dict[str, Any]]] = {}
        with gzip.open(path, "rt", encoding="utf-8") as handle:
            for line in tqdm(
                handle,
                desc=f"load {path.name}",
                unit="query",
            ):
                row = json.loads(line)
                query_id = str(row["query_id"])
                if query_id in rows:
                    raise RuntimeError(f"Duplicate ranking ID: {query_id}")
                rows[query_id] = [
                    {"docid": str(docid), "rank": rank}
                    for rank, docid in enumerate(row["docids"], start=1)
                ]
        return rows


    if reuse_rankings:
        hits_by_query_id = read_rankings(rankings_path)
    else:
        if retriever is None:
            raise RuntimeError("Retriever was not loaded.")
        unique_query_rows = query_artifacts[
            ["query_id", "query"]
        ].drop_duplicates("query_id")
        retrieval_result = run_query_retrieval_pipeline(
            queries=unique_query_rows,
            retriever=retriever,
            query_id_column="query_id",
            query_column="query",
            top_k=RETRIEVAL_TOP_K,
            batch_size=RETRIEVAL_BATCH_SIZE,
            measure_latency=False,
            progress=True,
            progress_desc=f"{BACKEND} full dev",
        )
        hits_by_query_id = retrieval_result.hits_by_query_id
        write_rankings(rankings_path, hits_by_query_id)
        write_json(
            retrieval_manifest_path,
            {
                "schema_version": 1,
                "backend": BACKEND,
                "d_no_history_query_rule": D_NO_HISTORY_QUERY_RULE,
                "backend_identity": backend_identity,
                "query_bundle_sha256": bundle_sha256,
                "query_count": len(unique_query_rows),
                "top_k": RETRIEVAL_TOP_K,
                "rankings_sha256": sha256_file(rankings_path),
                "complete": True,
            },
        )

    if set(hits_by_query_id) != set(query_artifacts["query_id"]):
        raise RuntimeError("Rankings and query bundle are not aligned.")
    if retriever is not None:
        close_retriever = getattr(retriever, "close", None)
        if callable(close_retriever):
            close_retriever()
        del retriever
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()


The completed metric artifacts are reused, so the underlying rankings do not need to be loaded and retrieval is not repeated.

### Evaluating retrieval routes

The stored rankings are converted into MRR, nDCG, and recall. Individual `I`, `R`, and `D` routes measure each retrieval view, while their RRF10 combinations test whether the views retrieve complementary passages. `IterCQR-1` and `IterCQR-G+2S-RRF10` provide the corresponding single-query and sampled-query baselines. Query-level metrics are retained for the paired bootstrap comparisons that follow.

In [12]:
if REUSE_COMPLETE_ANALYSIS:
    rocc_view_sets = {
        "I": ("I",), "R": ("R",), "D": ("D",),
        "I+R": ("I", "R"), "I+D": ("I", "D"),
        "R+D": ("R", "D"), "I+R+D": ("I", "R", "D"),
    }
    itercqr_view_sets = {
        "IterCQR-1": ("greedy",),
        "IterCQR-G+2S-RRF10": ("greedy", "sample_1", "sample_2"),
    }
    route_metrics = pd.read_csv(
        backend_dir / "route_metrics_by_query.csv"
    )
    route_summary = pd.read_csv(backend_dir / "route_summary.csv")
    display(
        route_summary.loc[route_summary["budget"].eq(PRIMARY_BUDGET)]
        .sort_values(["system", "arm"], kind="mergesort").round(4)
    )
else:
    rocc_view_sets = {
        "I": ("I",),
        "R": ("R",),
        "D": ("D",),
        "I+R": ("I", "R"),
        "I+D": ("I", "D"),
        "R+D": ("R", "D"),
        "I+R+D": ("I", "R", "D"),
    }
    itercqr_view_sets = {
        "IterCQR-1": ("greedy",),
        "IterCQR-G+2S-RRF10": ("greedy", "sample_1", "sample_2"),
    }
    rocc_route_metrics = evaluate_fixed_rrf_views(
        query_views=query_artifacts.loc[
            query_artifacts["family"].eq("rocc")
        ],
        hits_by_query_id=hits_by_query_id,
        gold_by_sample=gold_by_sample,
        view_sets=rocc_view_sets,
        group_columns=("system", "budget"),
        rrf_k=RRF_K,
        top_k=RETRIEVAL_TOP_K,
        ks=EVAL_KS,
        measure_latency=False,
        progress=True,
    )
    itercqr_route_metrics = evaluate_fixed_rrf_views(
        query_views=query_artifacts.loc[
            query_artifacts["family"].eq("itercqr")
        ],
        hits_by_query_id=hits_by_query_id,
        gold_by_sample=gold_by_sample,
        view_sets=itercqr_view_sets,
        group_columns=("system", "budget"),
        rrf_k=RRF_K,
        top_k=RETRIEVAL_TOP_K,
        ks=EVAL_KS,
        measure_latency=False,
        progress=True,
    )
    route_metrics = pd.concat(
        [rocc_route_metrics, itercqr_route_metrics],
        ignore_index=True,
    )
    route_summary = frame_summary(
        route_metrics,
        ("system", "budget", "arm"),
    )
    write_frame(backend_dir / "route_metrics_by_query.csv", route_metrics)
    write_frame(backend_dir / "route_summary.csv", route_summary)
    display(
        route_summary.loc[route_summary["budget"].eq(PRIMARY_BUDGET)]
        .sort_values(["system", "arm"], kind="mergesort")
        .round(4)
    )


,system,budget,arm,n,MRR,nDCG@3,R@10,R@100,R@1000
0,bm25_treatment_e6,64,D,2514,0.1687,0.1530,0.3039,0.4972,0.6516
1,bm25_treatment_e6,64,I,2514,0.2116,0.2014,0.3429,0.4924,0.6146
2,bm25_treatment_e6,64,I+D,2514,0.2189,0.2009,0.3858,0.5879,0.7232
3,bm25_treatment_e6,64,I+R,2514,0.2398,0.2247,0.4113,0.6118,0.7589
4,bm25_treatment_e6,64,I+R+D,2514,0.2435,0.2278,0.4220,0.6265,0.7701
5,bm25_treatment_e6,64,R,2514,0.2364,0.2238,0.3890,0.5887,0.7295
6,bm25_treatment_e6,64,R+D,2514,0.2242,0.2072,0.4041,0.6086,0.7538
28,imitation_control_e6,64,D,2514,0.1716,0.1552,0.3015,0.5032,0.6527
29,imitation_control_e6,64,I,2514,0.2116,0.2014,0.3429,0.4924,0.6146
30,imitation_control_e6,64,I+D,2514,0.2239,0.2075,0.3958,0.5891,0.7307


**BM25.** At B64, the base selector gives 0.1446 MRR for `I`, 0.1598 for `R`, 0.1570 for `D`, and 0.1846 for `I+R+D`. IterCQR-G+2S reaches 0.1532. The final fusion therefore exceeds the matched-budget sampled-rewrite baseline by 0.0314 MRR. Treatment gives 0.1602 for `R` and 0.1799 for `I+R+D`, compared with 0.1529 and 0.1763 for the imitation control.

**ANCE.** At B64, the base selector gives 0.2116 MRR for `I`, 0.2440 for `R`, 0.1741 for `D`, and 0.2463 for `I+R+D`. IterCQR-G+2S reaches 0.2154. The final fusion therefore exceeds the matched-budget sampled-rewrite baseline by 0.0309 MRR. Treatment gives 0.2364 for `R` and 0.2435 for `I+R+D`, compared with 0.2355 and 0.2426 for the imitation control.

These figures come from the frozen `results/07a_topiocqa_final/backends/{bm25,ance}/` route summaries and paired comparisons. The final result tables show both backends separately. The older experiment cells retain the backend of their last execution.

### Paired bootstrap comparisons

Mean scores alone do not show whether an observed difference is stable across queries. For each predefined comparison, the left and right routes are aligned on the same 2,514 queries and their metric difference is calculated per query. These paired differences are resampled 10,000 times to obtain a mean delta and 95% confidence interval.

The comparisons test three questions: whether pretrained `I+R+D` exceeds the IterCQR baselines, whether BM25 post-training improves over the imitation control and pretrained selector, and whether B64 fusion remains competitive with IterCQR sampling at B512. Separate `I+R+D`, `R`, and `D` contrasts identify where any post-training effect originates.

In [13]:
if REUSE_COMPLETE_ANALYSIS:
    primary_comparisons = pd.read_csv(
        backend_dir / "primary_comparisons.csv"
    )
    display(
        primary_comparisons.loc[
            primary_comparisons["metric"].eq("MRR")
        ].round(4)
    )
else:
    comparison_specs = {
        "pretrained_ird_minus_itercqr1": (
            ("pretrained", 64, "I+R+D"),
            ("itercqr", 64, "IterCQR-1"),
        ),
        "pretrained_ird_minus_itercqr_g2s": (
            ("pretrained", 64, "I+R+D"),
            ("itercqr", 64, "IterCQR-G+2S-RRF10"),
        ),
        "treatment_ird_minus_control_ird": (
            ("bm25_treatment_e6", 64, "I+R+D"),
            ("imitation_control_e6", 64, "I+R+D"),
        ),
        "treatment_ird_minus_pretrained_ird": (
            ("bm25_treatment_e6", 64, "I+R+D"),
            ("pretrained", 64, "I+R+D"),
        ),
        "control_ird_minus_pretrained_ird": (
            ("imitation_control_e6", 64, "I+R+D"),
            ("pretrained", 64, "I+R+D"),
        ),
        "treatment_r_minus_control_r": (
            ("bm25_treatment_e6", 64, "R"),
            ("imitation_control_e6", 64, "R"),
        ),
        "treatment_r_minus_pretrained_r": (
            ("bm25_treatment_e6", 64, "R"),
            ("pretrained", 64, "R"),
        ),
        "control_r_minus_pretrained_r": (
            ("imitation_control_e6", 64, "R"),
            ("pretrained", 64, "R"),
        ),
        "treatment_d_minus_control_d": (
            ("bm25_treatment_e6", 64, "D"),
            ("imitation_control_e6", 64, "D"),
        ),
        "treatment_d_minus_pretrained_d": (
            ("bm25_treatment_e6", 64, "D"),
            ("pretrained", 64, "D"),
        ),
        "control_d_minus_pretrained_d": (
            ("imitation_control_e6", 64, "D"),
            ("pretrained", 64, "D"),
        ),
        "pretrained_ird_b64_minus_itercqr_g2s_b512": (
            ("pretrained", 64, "I+R+D"),
            ("itercqr", 512, "IterCQR-G+2S-RRF10"),
        ),
    }


    def metric_series(spec: tuple[str, int, str], metric: str) -> pd.Series:
        system, budget, arm = spec
        selected = route_metrics.loc[
            route_metrics["system"].eq(system)
            & route_metrics["budget"].eq(int(budget))
            & route_metrics["arm"].eq(arm),
            ["sample_id", metric],
        ]
        if len(selected) != len(sample_ids):
            raise RuntimeError(f"Incomplete route: {spec}")
        return selected.set_index("sample_id")[metric].sort_index()


    primary_comparison_rows = []
    for comparison_index, comparison in enumerate(
        PRIMARY_COMPARISON_ORDER
    ):
        left_spec, right_spec = comparison_specs[comparison]
        for metric_index, metric in enumerate(METRIC_ORDER):
            left = metric_series(left_spec, metric)
            right = metric_series(right_spec, metric)
            left, right = left.align(right, join="inner")
            if len(left) != len(sample_ids):
                raise RuntimeError(f"Incomplete pairing: {comparison}")
            deltas = left.to_numpy() - right.to_numpy()
            seed = 1000 + comparison_index * len(METRIC_ORDER) + metric_index
            ci_low, ci_high = paired_bootstrap_mean_ci(
                deltas,
                seed=seed,
                replicates=BOOTSTRAP_REPLICATES,
                progress=True,
                progress_desc=f"bootstrap {comparison} {metric}",
            )
            primary_comparison_rows.append(
                {
                    "comparison": comparison,
                    "metric": metric,
                    "left_system": left_spec[0],
                    "left_budget": left_spec[1],
                    "left_arm": left_spec[2],
                    "right_system": right_spec[0],
                    "right_budget": right_spec[1],
                    "right_arm": right_spec[2],
                    "n": len(deltas),
                    "delta": float(deltas.mean()),
                    "ci95_low": ci_low,
                    "ci95_high": ci_high,
                    "seed": seed,
                    "replicates": BOOTSTRAP_REPLICATES,
                }
            )

    primary_comparisons = pd.DataFrame(primary_comparison_rows)
    write_frame(backend_dir / "primary_comparisons.csv", primary_comparisons)
    display(
        primary_comparisons.loc[
            primary_comparisons["metric"].eq("MRR")
        ].round(4)
    )


,comparison,metric,left_system,left_budget,left_arm,right_system,right_budget,right_arm,n,delta,ci95_low,ci95_high,seed,replicates
0,pretrained_ird_minus_itercqr1,MRR,pretrained,64,I+R+D,itercqr,64,IterCQR-1,2514,0.0347,0.0251,0.0445,1000,10000
5,pretrained_ird_minus_itercqr_g2s,MRR,pretrained,64,I+R+D,itercqr,64,IterCQR-G+2S-RRF10,2514,0.0309,0.0213,0.0409,1005,10000
10,treatment_ird_minus_control_ird,MRR,bm25_treatment_e6,64,I+R+D,imitation_control_e6,64,I+R+D,2514,0.0010,-0.0065,0.0083,1010,10000
15,treatment_ird_minus_pretrained_ird,MRR,bm25_treatment_e6,64,I+R+D,pretrained,64,I+R+D,2514,-0.0028,-0.0100,0.0046,1015,10000
20,control_ird_minus_pretrained_ird,MRR,imitation_control_e6,64,I+R+D,pretrained,64,I+R+D,2514,-0.0037,-0.0095,0.0022,1020,10000
25,treatment_r_minus_control_r,MRR,bm25_treatment_e6,64,R,imitation_control_e6,64,R,2514,0.0010,-0.0084,0.0101,1025,10000
30,treatment_r_minus_pretrained_r,MRR,bm25_treatment_e6,64,R,pretrained,64,R,2514,-0.0075,-0.0165,0.0016,1030,10000
35,control_r_minus_pretrained_r,MRR,imitation_control_e6,64,R,pretrained,64,R,2514,-0.0085,-0.0156,-0.0015,1035,10000
40,treatment_d_minus_control_d,MRR,bm25_treatment_e6,64,D,imitation_control_e6,64,D,2514,-0.0029,-0.0111,0.0050,1040,10000
45,treatment_d_minus_pretrained_d,MRR,bm25_treatment_e6,64,D,pretrained,64,D,2514,-0.0055,-0.0132,0.0021,1045,10000


**BM25.** Base B64 `I+R+D` versus the single B64 IterCQR rewrite gives +0.0400 MRR, 95% CI [+0.0314, +0.0486]. The comparison with B64 IterCQR-G+2S gives +0.0314 MRR, 95% CI [+0.0232, +0.0400]. Against B512 IterCQR-G+2S, the B64 fusion gives +0.0037 MRR, 95% CI [-0.0050, +0.0125]. These are paired comparisons on the same queries; a positive interval excludes zero in favor of the left-hand system.

**ANCE.** Base B64 `I+R+D` versus the single B64 IterCQR rewrite gives +0.0347 MRR, 95% CI [+0.0251, +0.0445]. The comparison with B64 IterCQR-G+2S gives +0.0309 MRR, 95% CI [+0.0213, +0.0409]. Against B512 IterCQR-G+2S, the B64 fusion gives -0.0161 MRR, 95% CI [-0.0263, -0.0064]. These are paired comparisons on the same queries; a positive interval excludes zero in favor of the left-hand system.

### CRF decoding analysis

A CRF assigns probabilities to complete history-selection sequences. Viterbi returns the single most probable sequence, while FFBS samples additional sequences from this distribution. These alternatives may retain relevant context omitted by Viterbi.

The analysis evaluates three B64 `R`-only variants. `Viterbi` uses one selected history, `2-FFBS-RRF10` fuses the rankings from two sampled histories, and `Viterbi+2-FFBS-RRF10` fuses Viterbi with both samples. Every selected history is rewritten by IterCQR and retrieved through the same backend. `I` and `D` are excluded so differences are attributable to CRF decoding.

All three variants are evaluated for the pretrained, imitation-control, and BM25-treatment checkpoints. Query-level MRR, nDCG, and recall are summarized over 2,514 queries. Pairwise checkpoint differences are evaluated with 10,000 paired bootstrap draws. This determines whether CRF uncertainty contains complementary retrieval evidence and whether post-training changes the usefulness of that uncertainty.

In [14]:
if REUSE_COMPLETE_ANALYSIS:
    crf_decoder_metrics = pd.read_csv(
        backend_dir / "crf_decoder_metrics_by_query.csv"
    )
    crf_decoder_summary = pd.read_csv(
        backend_dir / "crf_decoder_summary.csv"
    )
    crf_decoder_comparisons = pd.read_csv(
        backend_dir / "crf_decoder_comparisons.csv"
    )
    display(crf_decoder_summary.round(4))
else:
    crf_view_sets = {
        "Viterbi": ("viterbi",),
        "2-FFBS-RRF10": ("ffbs_1", "ffbs_2"),
        "Viterbi+2-FFBS-RRF10": (
            "viterbi",
            "ffbs_1",
            "ffbs_2",
        ),
    }
    crf_decoder_metrics = evaluate_fixed_rrf_views(
        query_views=query_artifacts.loc[
            query_artifacts["family"].eq("crf")
        ],
        hits_by_query_id=hits_by_query_id,
        gold_by_sample=gold_by_sample,
        view_sets=crf_view_sets,
        group_columns=("system", "budget"),
        rrf_k=RRF_K,
        top_k=RETRIEVAL_TOP_K,
        ks=EVAL_KS,
        measure_latency=False,
        progress=True,
    )
    crf_decoder_summary = frame_summary(
        crf_decoder_metrics,
        ("system", "budget", "arm"),
    )

    decoder_comparison_rows = []
    decoder_system_pairs = (
        ("bm25_treatment_e6", "imitation_control_e6"),
        ("bm25_treatment_e6", "pretrained"),
        ("imitation_control_e6", "pretrained"),
    )
    comparison_index = 0
    for arm in crf_view_sets:
        for left_system, right_system in decoder_system_pairs:
            for metric_index, metric in enumerate(METRIC_ORDER):
                selected = crf_decoder_metrics.loc[
                    crf_decoder_metrics["arm"].eq(arm)
                    & crf_decoder_metrics["system"].isin(
                        [left_system, right_system]
                    ),
                    ["sample_id", "system", metric],
                ].pivot(index="sample_id", columns="system", values=metric)
                deltas = (
                    selected[left_system] - selected[right_system]
                ).to_numpy()
                seed = 3000 + comparison_index * len(METRIC_ORDER) + metric_index
                ci_low, ci_high = paired_bootstrap_mean_ci(
                    deltas,
                    seed=seed,
                    replicates=BOOTSTRAP_REPLICATES,
                    progress=True,
                    progress_desc=(
                        f"bootstrap {arm} {left_system}-{right_system} {metric}"
                    ),
                )
                decoder_comparison_rows.append(
                    {
                        "arm": arm,
                        "metric": metric,
                        "comparison": f"{left_system} - {right_system}",
                        "left_system": left_system,
                        "right_system": right_system,
                        "n": len(deltas),
                        "delta": float(deltas.mean()),
                        "ci95_low": ci_low,
                        "ci95_high": ci_high,
                        "seed": seed,
                        "replicates": BOOTSTRAP_REPLICATES,
                    }
                )
            comparison_index += 1

    crf_decoder_comparisons = pd.DataFrame(decoder_comparison_rows)
    write_frame(
        backend_dir / "crf_decoder_metrics_by_query.csv",
        crf_decoder_metrics,
    )
    write_frame(
        backend_dir / "crf_decoder_summary.csv",
        crf_decoder_summary,
    )
    write_frame(
        backend_dir / "crf_decoder_comparisons.csv",
        crf_decoder_comparisons,
    )
    display(crf_decoder_summary.round(4))


,system,budget,arm,n,MRR,nDCG@3,R@10,R@100,R@1000
0,bm25_treatment_e6,64,2-FFBS-RRF10,2514,0.2446,0.2289,0.4141,0.6050,0.7454
1,bm25_treatment_e6,64,Viterbi,2514,0.2364,0.2238,0.3890,0.5887,0.7295
2,bm25_treatment_e6,64,Viterbi+2-FFBS-RRF10,2514,0.2509,0.2329,0.4228,0.6269,0.7709
3,imitation_control_e6,64,2-FFBS-RRF10,2514,0.2372,0.2212,0.4109,0.6102,0.7478
4,imitation_control_e6,64,Viterbi,2514,0.2355,0.2193,0.4045,0.5863,0.7208
5,imitation_control_e6,64,Viterbi+2-FFBS-RRF10,2514,0.2435,0.2267,0.4137,0.6309,0.7641
6,pretrained,64,2-FFBS-RRF10,2514,0.2409,0.2238,0.4161,0.6086,0.7494
7,pretrained,64,Viterbi,2514,0.2440,0.2315,0.4105,0.5915,0.7263
8,pretrained,64,Viterbi+2-FFBS-RRF10,2514,0.2496,0.2339,0.4260,0.6321,0.7721


**BM25.** For every checkpoint, fusing Viterbi with both FFBS samples produces the highest MRR. The combined route reaches 0.1811 for BM25 treatment, 0.1764 for pretrained, and 0.1672 for the imitation control. BM25 treatment shows supported gains over the imitation control for `2-FFBS-RRF10` (+0.0093 MRR) and the combined route (+0.0139). Its differences from pretrained remain unsupported. Post-training therefore improves CRF sampling relative to continued imitation but does not establish an advantage over the original pretrained selector.

**ANCE (retriever OOD).** `Viterbi+2-FFBS-RRF10` again provides the highest point estimate for every checkpoint: 0.2509 for treatment, 0.2496 for pretrained, and 0.2435 for control. However, treatment does not show a supported MRR gain over control for either `2-FFBS-RRF10` (+0.0074, 95% CI [−0.0018, +0.0169]) or the combined decoder (+0.0074, [−0.0012, +0.0161]).

**Within-checkpoint comparison used in the thesis.** Each fusion is compared with the single Viterbi rewrite from the same checkpoint. Viterbi plus two FFBS samples improves all three checkpoints with BM25. For $f_{\mathrm{base}}$, MRR rises from 0.1598 to 0.1764, a gain of 0.0166 with a 95% interval of [0.0108, 0.0223]. With ANCE, both post-trained checkpoints improve, but the base student’s gain of 0.0056 has an interval of [−0.0007, 0.0118].

**Cross-backend interpretation.** Sampled histories can provide complementary evidence beyond the single Viterbi sequence. The post-training advantage over continued imitation is supported only under BM25, not under ANCE, and neither backend establishes a treatment advantage over pretrained.

### Compression and budget effects

ROCC concentrates retrieval evidence before the token budget is applied. Compression rates quantify the retained history. Input and rewrite identity show how often additional capacity changes the IterCQR query. Paired MRR differences with 95% bootstrap intervals measure the resulting retrieval effect for `R` and `D`.

In [15]:
BUDGET_COMPARISON_COLUMNS = (
    "system",
    "arm",
    "comparison",
    "n",
    "mrr_delta",
    "ci95_low",
    "ci95_high",
    "seed",
    "replicates",
)

if REUSE_COMPLETE_ANALYSIS:
    compression_summary = pd.read_csv(
        backend_dir / "compression_summary.csv"
    )
    budget_identity = pd.read_csv(backend_dir / "budget_identity.csv")
    budget_mrr_comparisons = pd.read_csv(
        backend_dir / budget_comparison_file
    ).loc[:, BUDGET_COMPARISON_COLUMNS]
    display(compression_summary.round(4))
    display(budget_identity.round(4))
    display(budget_mrr_comparisons.round(4))
else:
    original_words_by_sample = {
        str(row["sample_id"]): history_word_count(row["history"])
        for row in dev_rows
    }
    selected_words_by_system = {
        system: {
            sample_id: history_word_count(history)
            for sample_id, history in selector_predictions[system][
                "viterbi"
            ].items()
        }
        for system in SYSTEM_ORDER
    }

    compression_rows = []
    identity_rows = []
    for system in SYSTEM_ORDER:
        arm = f"{system}_R"
        for budget in BUDGETS:
            selected = main_rewrites.loc[
                main_rewrites["arm"].eq(arm)
                & main_rewrites["budget"].eq(budget)
            ]
            original_words = sum(original_words_by_sample.values())
            selected_words = sum(selected_words_by_system[system].values())
            keep_rate = (
                selected_words / original_words if original_words else 0.0
            )
            compression_rows.append(
                {
                    "system": system,
                    "budget": int(budget),
                    "queries": int(selected["sample_id"].nunique()),
                    "original_history_words": int(original_words),
                    "selected_history_words": int(selected_words),
                    "mean_selected_history_words": float(
                        np.mean(list(selected_words_by_system[system].values()))
                    ),
                    "keep_rate": float(keep_rate),
                    "compression_rate": float(1.0 - keep_rate),
                    "mean_t5_input_tokens": float(
                        selected["input_length"].mean()
                    ),
                    "truncation_rate": float(
                        selected["was_truncated"].mean()
                    ),
                    "empty_selection_count": int(
                        sum(
                            selected_words_by_system[system][sample_id] == 0
                            for sample_id in sample_ids
                            if depth_by_sample[sample_id] >= 1
                        )
                    ),
                }
            )
        for lower, upper in zip(BUDGETS[:-1], BUDGETS[1:], strict=True):
            lower_rows = main_rewrites.loc[
                main_rewrites["arm"].eq(arm)
                & main_rewrites["budget"].eq(lower),
                ["sample_id", "input_key", "rewrite_norm"],
            ].set_index("sample_id")
            upper_rows = main_rewrites.loc[
                main_rewrites["arm"].eq(arm)
                & main_rewrites["budget"].eq(upper),
                ["sample_id", "input_key", "rewrite_norm"],
            ].set_index("sample_id")
            paired = lower_rows.join(
                upper_rows,
                lsuffix="_lower",
                rsuffix="_upper",
                how="inner",
                validate="one_to_one",
            )
            identity_rows.append(
                {
                    "system": system,
                    "comparison": f"B{upper}-B{lower}",
                    "n": len(paired),
                    "identical_input_share": float(
                        paired["input_key_lower"].eq(
                            paired["input_key_upper"]
                        ).mean()
                    ),
                    "identical_rewrite_share": float(
                        paired["rewrite_norm_lower"].eq(
                            paired["rewrite_norm_upper"]
                        ).mean()
                    ),
                }
            )

    compression_summary = pd.DataFrame(compression_rows)
    budget_identity = pd.DataFrame(identity_rows)
    budget_comparison_rows = []
    budget_pairs = tuple(zip(BUDGETS[:-1], BUDGETS[1:], strict=True))
    for system_index, system in enumerate(SYSTEM_ORDER):
        for arm_index, arm in enumerate(("R", "D")):
            for pair_index, (lower, upper) in enumerate(budget_pairs):
                lower_values = metric_series((system, lower, arm), "MRR")
                upper_values = metric_series((system, upper, arm), "MRR")
                lower_values, upper_values = lower_values.align(
                    upper_values,
                    join="inner",
                )
                deltas = upper_values.to_numpy() - lower_values.to_numpy()
                seed = (
                    2000
                    + system_index * 6
                    + arm_index * len(budget_pairs)
                    + pair_index
                )
                ci_low, ci_high = paired_bootstrap_mean_ci(
                    deltas,
                    seed=seed,
                    replicates=BOOTSTRAP_REPLICATES,
                    progress=True,
                    progress_desc=(
                        f"budget {system} {arm} B{upper}-B{lower}"
                    ),
                )
                budget_comparison_rows.append(
                    {
                        "system": system,
                        "arm": arm,
                        "comparison": f"B{upper}-B{lower}",
                        "n": len(deltas),
                        "mrr_delta": float(deltas.mean()),
                        "ci95_low": ci_low,
                        "ci95_high": ci_high,
                        "seed": seed,
                        "replicates": BOOTSTRAP_REPLICATES,
                    }
                )

    budget_mrr_comparisons = pd.DataFrame(
        budget_comparison_rows,
        columns=BUDGET_COMPARISON_COLUMNS,
    )
    write_frame(backend_dir / "compression_summary.csv", compression_summary)
    write_frame(backend_dir / "budget_identity.csv", budget_identity)
    write_frame(
        backend_dir / "budget_mrr_comparisons.csv",
        budget_mrr_comparisons,
    )
    display(compression_summary.round(4))
    display(budget_identity.round(4))
    display(budget_mrr_comparisons.round(4))


,system,budget,queries,original_history_words,selected_history_words,mean_selected_history_words,keep_rate,compression_rate,mean_t5_input_tokens,truncation_rate,empty_selection_count
0,pretrained,64,2514,251706,42679,16.9765,0.1696,0.8304,40.8894,0.1356,0
1,pretrained,128,2514,251706,42679,16.9765,0.1696,0.8304,41.8628,0.0728,0
2,pretrained,256,2514,251706,42679,16.9765,0.1696,0.8304,41.8755,0.0728,0
3,pretrained,512,2514,251706,42679,16.9765,0.1696,0.8304,41.8755,0.0728,0
4,imitation_control_e6,64,2514,251706,36620,14.5664,0.1455,0.8545,37.5569,0.0903,0
5,imitation_control_e6,128,2514,251706,36620,14.5664,0.1455,0.8545,38.0461,0.0573,0
6,imitation_control_e6,256,2514,251706,36620,14.5664,0.1455,0.8545,38.0533,0.0573,0
7,imitation_control_e6,512,2514,251706,36620,14.5664,0.1455,0.8545,38.0533,0.0573,0
8,bm25_treatment_e6,64,2514,251706,51359,20.4292,0.2040,0.7960,46.4220,0.2884,0
9,bm25_treatment_e6,128,2514,251706,51359,20.4292,0.2040,0.7960,50.9467,0.0887,0


,system,comparison,n,identical_input_share,identical_rewrite_share
0,pretrained,B128-B64,2514,0.9045,0.9252
1,pretrained,B256-B128,2514,0.9996,0.9996
2,pretrained,B512-B256,2514,1.0000,1.0000
3,imitation_control_e6,B128-B64,2514,0.9507,0.9622
4,imitation_control_e6,B256-B128,2514,0.9996,0.9996
5,imitation_control_e6,B512-B256,2514,1.0000,1.0000
6,bm25_treatment_e6,B128-B64,2514,0.7399,0.7963
7,bm25_treatment_e6,B256-B128,2514,0.9936,0.9944
8,bm25_treatment_e6,B512-B256,2514,1.0000,1.0000


,system,arm,comparison,n,mrr_delta,ci95_low,ci95_high,seed,replicates
0,pretrained,R,B128-B64,2514,-0.0003,-0.0027,0.0022,2000,10000
1,pretrained,R,B256-B128,2514,0.0000,0.0000,0.0000,2001,10000
2,pretrained,R,B512-B256,2514,0.0000,0.0000,0.0000,2002,10000
3,pretrained,D,B128-B64,2514,-0.0003,-0.0018,0.0012,2003,10000
4,pretrained,D,B256-B128,2514,0.0000,0.0000,0.0000,2004,10000
5,pretrained,D,B512-B256,2514,0.0000,0.0000,0.0000,2005,10000
6,imitation_control_e6,R,B128-B64,2514,0.0004,-0.0011,0.0019,2006,10000
7,imitation_control_e6,R,B256-B128,2514,-0.0000,-0.0000,0.0000,2007,10000
8,imitation_control_e6,R,B512-B256,2514,0.0000,0.0000,0.0000,2008,10000
9,imitation_control_e6,D,B128-B64,2514,0.0006,-0.0001,0.0014,2009,10000


The pretrained selector retains 16.96% of the original history, the imitation control 14.55%, and the BM25 treatment 20.40%. All history-bearing queries receive non-empty selections. At B64, treatment has the highest truncation rate at 28.84%, including truncation by either the per-segment limit or the total input budget. Increasing its budget to B128 changes 26.01% of the serialized inputs and 20.37% of the IterCQR rewrites. These selection, serialization, and rewrite statistics are shared by both retrieval backends.

**BM25.** The treatment `R` route gains 0.0045 MRR from B64 to B128 with a 95% confidence interval of [+0.0007, +0.0085]. The remaining treatment routes and both comparison systems show absolute B64-to-B128 differences below 0.001. From B128 to B256, every absolute difference is at most 0.0001. B256 and B512 produce identical MRR values. The BM25 retrieval effect of additional treatment context is therefore concentrated in `R` between B64 and B128.

**ANCE (retriever OOD).** For treatment, increasing the budget from B64 to B128 improves `R` by +0.0075 MRR with a 95% CI of [+0.0028, +0.0124] and `D` by +0.0037 with [+0.0009, +0.0067]. For pretrained and control, the corresponding changes remain practically small. From B128 to B256, all absolute changes are at most 0.0003, and B256 and B512 are identical. ANCE therefore also shows saturation after B128, although treatment benefits from the additional B128 context in both `R` and `D`.

**Cross-backend interpretation.** Input-level saturation is shared because both retrievers receive the same frozen queries. Retrieval quality likewise stops changing materially beyond B128, while the treatment-specific B64-to-B128 gain reaches `D` only under ANCE.

### Retrieval performance by history depth

The scope for history selection increases with the number of previous turns. B64 performance is therefore summarized separately for IterCQR-1 and the three `I+R+D` systems across fixed depth bins. `d00` and `d01` cover queries with zero or one previous turn. The remaining bins reuse the depth boundaries established in NB03.

The strata locate where the aggregate retrieval differences occur and whether they increase in deeper conversations.

In [16]:
if REUSE_COMPLETE_ANALYSIS:
    strata_summary = pd.read_csv(
        backend_dir / "depth_strata_summary.csv"
    )
    display(strata_summary.round(4))
else:
    def depth_bin(depth: int) -> str:
        if int(depth) == 0:
            return "d00"
        if int(depth) == 1:
            return "d01"
        for label in ORACLE_GATE_DEPTH_BIN_ORDER:
            lower, upper = ORACLE_GATE_DEPTH_BOUNDS[label]
            if int(depth) >= lower and (
                upper is None or int(depth) <= upper
            ):
                return str(label)
        raise ValueError(f"Uncovered history depth: {depth}")


    central_specs = (
        ("itercqr", "IterCQR-1"),
        ("pretrained", "I+R+D"),
        ("imitation_control_e6", "I+R+D"),
        ("bm25_treatment_e6", "I+R+D"),
    )
    strata_parts = []
    for system, arm in central_specs:
        selected = route_metrics.loc[
            route_metrics["system"].eq(system)
            & route_metrics["budget"].eq(PRIMARY_BUDGET)
            & route_metrics["arm"].eq(arm)
        ].copy()
        selected["depth_bin"] = selected["sample_id"].map(
            lambda sample_id: depth_bin(depth_by_sample[str(sample_id)])
        )
        strata_parts.append(selected)
    strata_metrics = pd.concat(strata_parts, ignore_index=True)
    strata_summary = frame_summary(
        strata_metrics,
        ("system", "arm", "depth_bin"),
    )
    depth_order = (
        "d00",
        "d01",
        *ORACLE_GATE_DEPTH_BIN_ORDER,
    )
    strata_summary["depth_bin"] = pd.Categorical(
        strata_summary["depth_bin"],
        categories=depth_order,
        ordered=True,
    )
    strata_summary = strata_summary.sort_values(
        ["system", "arm", "depth_bin"],
        kind="mergesort",
    )
    write_frame(backend_dir / "depth_strata_summary.csv", strata_summary)
    display(strata_summary.round(4))


,system,arm,depth_bin,n,MRR,nDCG@3,R@10,R@100,R@1000
0,bm25_treatment_e6,I+R+D,d00,205,0.3008,0.2866,0.5073,0.6878,0.8049
1,bm25_treatment_e6,I+R+D,d01,205,0.3645,0.3524,0.5756,0.7317,0.8439
2,bm25_treatment_e6,I+R+D,d02_04,615,0.2902,0.2769,0.4699,0.7073,0.8228
3,bm25_treatment_e6,I+R+D,d05_06,410,0.2366,0.2209,0.4146,0.6244,0.7537
4,bm25_treatment_e6,I+R+D,d07_08,410,0.2090,0.1909,0.3829,0.5659,0.7488
5,bm25_treatment_e6,I+R+D,d09_10,354,0.1986,0.1763,0.3616,0.5650,0.7232
6,bm25_treatment_e6,I+R+D,d11_14,305,0.1400,0.1261,0.3016,0.5115,0.7016
7,bm25_treatment_e6,I+R+D,d15_plus,10,0.1722,0.1631,0.3000,0.5000,0.6000
8,imitation_control_e6,I+R+D,d00,205,0.3008,0.2866,0.5073,0.6878,0.8049
9,imitation_control_e6,I+R+D,d01,205,0.3639,0.3572,0.5756,0.7366,0.8585


**BM25.** Without history, `I` and `R` have the same input and score 0.1564 MRR. This does not imply equality with direct retrieval or rank fusion: `I+R+D` scores 0.1685. At depths 2–4, base fusion exceeds the single IterCQR rewrite by +0.0396 MRR. At depths 5–6, base fusion exceeds the single IterCQR rewrite by +0.0567 MRR. At depths 7–8, base fusion exceeds the single IterCQR rewrite by +0.0630 MRR.

**ANCE.** Without history, `I` and `R` have the same input and score 0.3020 MRR. This does not imply equality with direct retrieval or rank fusion: `I+R+D` scores 0.3008. At depths 2–4, base fusion exceeds the single IterCQR rewrite by +0.0401 MRR. At depths 5–6, base fusion exceeds the single IterCQR rewrite by +0.0317 MRR. At depths 7–8, base fusion exceeds the single IterCQR rewrite by +0.0400 MRR.

### History-selection granularity (BM25-only supplemental ablation)

Token-level selection is a central design choice because it adds fine-grained supervision, decoding, and projection logic. Its value depends on whether exact token boundaries improve downstream retrieval beyond identifying the relevant question, answer, or dialogue turn.

The Viterbi selection of $f_{\mathrm{base}}$ is expanded to two coarser representations. The code label `token` corresponds to thesis granularity V3. `qa` corresponds to V2 and retains the complete query or answer field containing a selected word. `turn` corresponds to V1 and retains the complete query–answer turn. The thesis reports $R_{64}$ for these projections. Other routes and budgets below are supplemental results.

`R` measures how granularity affects the IterCQR rewrite, `D` measures its direct retrieval effect, and `I+R+D` measures its effect within the complete fusion. The fixed `I` route provides a common anchor. Mean T5 input length shows how much additional context each projection introduces.

A token-level advantage would justify precise span selection. Comparable Q/A or turn performance would show that identifying the relevant conversational unit captures most of the retrieval signal. Query-level MRR differences are paired by sample and evaluated with 10,000 bootstrap replicates for `R`, `D`, and `I+R+D`. This supplemental projection is evaluated only with BM25; no corresponding ANCE granularity artifacts exist.

In [17]:
# Persistent BM25 projection of the pretrained Viterbi selection.
# Token results come from NB07a. Q/A and turn results are computed
# when the subartifact is missing and subsequently loaded from cache.
if BACKEND == "bm25":
    expected_budgets = list(BUDGETS)
    expected_arms = ["I", "R", "D", "I+R+D"]
    token_routes = route_summary.loc[
        route_summary["system"].eq("pretrained")
        & route_summary["budget"].isin(expected_budgets)
        & route_summary["arm"].isin(expected_arms),
        ["budget", "arm", "n", "MRR"],
    ].copy()
    if (
        len(token_routes) != len(expected_budgets) * len(expected_arms)
        or set(token_routes["n"].astype(int)) != {2514}
    ):
        raise RuntimeError(
            "Incomplete token routes for the granularity table."
        )
    token_mrr = token_routes.pivot(
        index="budget", columns="arm", values="MRR"
    )

    from experiments.rocc.granularity_projection import (
        load_or_compute_topiocqa_bm25_granularity_projection,
    )

    projection_result = (
        load_or_compute_topiocqa_bm25_granularity_projection(
            result_dir=RESULT_DIR,
            backend_dir=backend_dir,
            rows=dev_rows,
            samples=dev_samples,
            sample_ids=sample_ids,
            depth_by_sample=depth_by_sample,
            gold_by_sample=gold_by_sample,
            population_sha256=population_sha256,
            selector_checkpoint_sha256=(
                EXPECTED_CHECKPOINT_HASHES["pretrained"]
            ),
            query_bundle_manifest=query_bundle_manifest,
            backend_identity=backend_identity,
            pipeline_config=PIPELINE_CONFIG,
            itercqr_model_dir=ITERCQR_MODEL_DIR,
            budgets=BUDGETS,
            rrf_k=RRF_K,
            retrieval_top_k=RETRIEVAL_TOP_K,
            eval_ks=EVAL_KS,
            progress=True,
        )
    )
    granularity_metrics_by_query = (
        projection_result.metrics_by_query
    )
    granularity_summary = projection_result.summary
    granularity_input_length_summary = (
        projection_result.input_length_summary
    )
    token_lengths = (
        granularity_input_length_summary.loc[
            granularity_input_length_summary["granularity"].eq(
                "token"
            ),
            [
                "budget",
                "mean_I_t5_input_tokens",
                "mean_R_t5_input_tokens",
            ],
        ].set_index("budget")
    )
    granularity_serialization_summary = (
        projection_result.serialization_summary
    )
    granularity_comparison_arms = ("R", "D", "I+R+D")
    granularity_comparison_specs = (
        ("Token − Q/A", "token", "qa"),
        ("Token − Turn", "token", "turn"),
    )
    granularity_comparison_rows = []
    for budget_index, budget in enumerate(expected_budgets):
        for arm_index, arm in enumerate(
            granularity_comparison_arms
        ):
            paired = (
                granularity_metrics_by_query.loc[
                    granularity_metrics_by_query["budget"].eq(budget)
                    & granularity_metrics_by_query["arm"].eq(arm),
                    ["sample_id", "granularity", "MRR"],
                ]
                .pivot(
                    index="sample_id",
                    columns="granularity",
                    values="MRR",
                )
                .sort_index()
            )
            if (
                len(paired) != len(sample_ids)
                or paired.isna().any().any()
            ):
                raise RuntimeError(
                    f"Incomplete granularity pairing: B{budget} {arm}"
                )
            for comparison_index, (
                comparison,
                left_granularity,
                right_granularity,
            ) in enumerate(granularity_comparison_specs):
                deltas = (
                    paired[left_granularity]
                    - paired[right_granularity]
                ).to_numpy()
                seed = (
                    4000
                    + budget_index
                    * len(granularity_comparison_arms)
                    * len(granularity_comparison_specs)
                    + arm_index * len(granularity_comparison_specs)
                    + comparison_index
                )
                ci_low, ci_high = paired_bootstrap_mean_ci(
                    deltas,
                    seed=seed,
                    replicates=BOOTSTRAP_REPLICATES,
                    progress=False,
                )
                granularity_comparison_rows.append(
                    {
                        "budget": int(budget),
                        "arm": arm,
                        "comparison": comparison,
                        "left_granularity": left_granularity,
                        "right_granularity": right_granularity,
                        "n": len(deltas),
                        "delta_mrr": float(deltas.mean()),
                        "ci95_low": ci_low,
                        "ci95_high": ci_high,
                        "ci95_excludes_zero": bool(
                            ci_low > 0.0 or ci_high < 0.0
                        ),
                        "seed": seed,
                        "replicates": BOOTSTRAP_REPLICATES,
                    }
                )
    granularity_mrr_comparisons = pd.DataFrame(
        granularity_comparison_rows
    )
    granularity_comparison_path = (
        backend_dir / "granularity_mrr_comparisons.csv"
    )
    granularity_comparison_relative = str(
        granularity_comparison_path.relative_to(backend_dir)
    )
    if REUSE_COMPLETE_ANALYSIS:
        bound_comparison_sha256 = existing_final_manifest.get(
            "files", {}
        ).get(granularity_comparison_relative)
        if bound_comparison_sha256 is not None and (
            not granularity_comparison_path.is_file()
            or sha256_file(granularity_comparison_path)
            != bound_comparison_sha256
        ):
            raise RuntimeError(
                "The bound granularity comparisons are missing "
                "or have drifted."
            )
    write_frame(
        granularity_comparison_path,
        granularity_mrr_comparisons,
    )
    granularity_comparison_sha256 = sha256_file(
        granularity_comparison_path
    )
    granularity_projection_manifest_path = (
        projection_result.manifest_path
    )
    granularity_projection_file_hashes = {
        str(path.relative_to(backend_dir)): sha256_file(path)
        for path in projection_result.data_paths
    }
    granularity_projection_extension = {
        "protocol": projection_result.manifest["protocol"],
        "manifest": str(
            projection_result.manifest_path.relative_to(backend_dir)
        ),
        "manifest_sha256": sha256_file(
            projection_result.manifest_path
        ),
    }
    if REUSE_COMPLETE_ANALYSIS:
        updated_manifest = dict(existing_final_manifest)
        updated_files = dict(updated_manifest.get("files", {}))
        recorded_projection = {
            key: value
            for key, value in updated_files.items()
            if key.startswith("granularity_projection/")
        }
        updated_extensions = dict(
            updated_manifest.get("extensions", {})
        )
        recorded_extension = updated_extensions.get(
            "granularity_projection"
        )
        recorded_comparison_sha256 = updated_files.get(
            granularity_comparison_relative
        )
        if (
            recorded_projection
            and recorded_projection
            != granularity_projection_file_hashes
        ):
            raise RuntimeError(
                "Granularity files differ in the main manifest."
            )
        if (
            recorded_extension is not None
            and recorded_extension
            != granularity_projection_extension
        ):
            raise RuntimeError(
                "Granularity extension differs in the main manifest."
            )
        if bool(recorded_projection) != (
            recorded_extension is not None
        ):
            raise RuntimeError(
                "Granularity is only partially bound to the main "
                "manifest."
            )
        if (
            recorded_comparison_sha256 is not None
            and recorded_comparison_sha256
            != granularity_comparison_sha256
        ):
            raise RuntimeError(
                "Granularity comparisons differ in the main "
                "manifest."
            )
        manifest_changed = False
        if not recorded_projection:
            updated_files.update(
                granularity_projection_file_hashes
            )
            updated_extensions["granularity_projection"] = (
                granularity_projection_extension
            )
            manifest_changed = True
        if recorded_comparison_sha256 is None:
            updated_files[granularity_comparison_relative] = (
                granularity_comparison_sha256
            )
            manifest_changed = True
        if manifest_changed:
            updated_manifest["files"] = dict(
                sorted(updated_files.items())
            )
            updated_manifest["extensions"] = updated_extensions
            write_json(backend_manifest_path, updated_manifest)
            existing_final_manifest = updated_manifest

    projected_summary = granularity_summary.set_index(
        ["granularity", "budget", "arm"]
    )
    projected_lengths = (
        granularity_input_length_summary.set_index(
            ["granularity", "budget"]
        )
    )
    projected = {}
    for level, granularity in (("Q/A", "qa"), ("Turn", "turn")):
        projected[level] = {}
        for budget in expected_budgets:
            projected[level][budget] = (
                float(
                    projected_summary.loc[
                        (granularity, budget, "R"), "MRR"
                    ]
                ),
                float(
                    projected_summary.loc[
                        (granularity, budget, "D"), "MRR"
                    ]
                ),
                float(
                    projected_summary.loc[
                        (granularity, budget, "I+R+D"), "MRR"
                    ]
                ),
                float(
                    projected_lengths.loc[
                        (granularity, budget),
                        "mean_R_t5_input_tokens",
                    ]
                ),
            )
    granularity_rows = []
    for budget in expected_budgets:
        i_mrr = float(token_mrr.loc[budget, "I"])
        i_tokens = float(
            token_lengths.loc[budget, "mean_I_t5_input_tokens"]
        )
        level_values = {
            "Token": (
                float(token_mrr.loc[budget, "R"]),
                float(token_mrr.loc[budget, "D"]),
                float(token_mrr.loc[budget, "I+R+D"]),
                float(
                    token_lengths.loc[
                        budget, "mean_R_t5_input_tokens"
                    ]
                ),
            ),
            "Q/A": projected["Q/A"][budget],
            "Turn": projected["Turn"][budget],
        }
        for level, (r_mrr, d_mrr, ird_mrr, r_tokens) in (
            level_values.items()
        ):
            reduction = 100.0 * (1.0 - r_tokens / i_tokens)
            token_text = (
                f"{i_tokens:.1f} → {r_tokens:.1f} "
                f"(−{reduction:.1f} %)"
            )
            granularity_rows.append(
                {
                    "Granularity": level,
                    "Budget": int(budget),
                    "I": i_mrr,
                    "R": r_mrr,
                    "D": d_mrr,
                    "I+R+D": ird_mrr,
                    "Mean T5 input tokens: I → R": token_text,
                }
            )

    granularity_projection_summary = pd.DataFrame(granularity_rows)
    display(
        granularity_projection_summary.style.format(
            {
                "I": lambda value: f"{value:.4f}",
                "R": lambda value: f"{value:.4f}",
                "D": lambda value: f"{value:.4f}",
                "I+R+D": lambda value: (
                    f"{value:.4f}"
                ),
            }
        ).hide(axis="index")
    )
    granularity_comparison_display = (
        granularity_mrr_comparisons.assign(
            estimate_and_ci=granularity_mrr_comparisons.apply(
                lambda row: (
                    f"{row['delta_mrr']:+.4f} "
                    f"[{row['ci95_low']:+.4f}, "
                    f"{row['ci95_high']:+.4f}]"
                ),
                axis=1,
            )
        )
        .set_index(["budget", "comparison", "arm"])[
            "estimate_and_ci"
        ]
        .unstack("arm")
        .reset_index()
    )
    granularity_comparison_display.columns.name = None
    granularity_comparison_display = (
        granularity_comparison_display.loc[
            :,
            [
                "budget",
                "comparison",
                *granularity_comparison_arms,
            ],
        ].rename(
            columns={
                "budget": "Budget",
                "comparison": "Comparison",
                "R": "R: ΔMRR [95% CI]",
                "D": "D: ΔMRR [95% CI]",
                "I+R+D": "I+R+D: ΔMRR [95% CI]",
            }
        )
    )
    display(granularity_comparison_display)
else:
    granularity_projection_summary = pd.DataFrame()
    granularity_mrr_comparisons = pd.DataFrame()
    granularity_projection_manifest_path = None
    granularity_projection_file_hashes = {}
    granularity_projection_extension = None
    display(
        Markdown(
            "**Granularity projection:** This supplemental analysis "
            "is reported only for the local BM25 run."
        )
    )


**Granularity projection:** This supplemental analysis is reported only for the local BM25 run.

**BM25 only.** All variants originate from the same Viterbi-selected words. V2 and V1 expand them before the input budget is applied. In the thesis comparison, $R_{64}$ reaches 0.1598 MRR at V3, 0.1550 at V2, and 0.1537 at V1. Both paired differences from V3 have intervals that include zero. Mean input length rises from 40.9 tokens at V3 to 50.1 at V2 and 53.3 at V1. The added context therefore lowers the point estimates without a statistically supported loss.

`D` sends the selected history directly to BM25. Every additional word can influence the sparse retrieval score. Q/A and turn expansion add neighboring terms that can change lexical matches. In these supplemental direct-route comparisons, token selection has a supported `D` advantage at every budget.

`R` first passes the selected history through IterCQR. The T5 rewriter can omit surrounding terms before retrieval. This is a possible explanation for the smaller `R` differences, whose confidence intervals all include zero. Precise token boundaries therefore provide no supported advantage when the rewriter alone determines the retrieval query.

`I` is identical across granularities. The fusion difference is driven by the balance between `R` and `D`. At B64, the token-level improvement in `D` is large enough to produce supported `I+R+D` gains over Q/A and turn selection. At larger budgets, the coarser `R` routes obtain slightly higher point estimates and partially offset the token-level `D` advantage. The remaining fusion differences are too small to exclude zero.

Token-level selection is therefore justified for the complete B64 BM25 system because it improves direct retrieval, preserves complementary evidence for fusion, and requires only 40.9 mean `R` input tokens. The results do not establish a token-level advantage for `R` alone or for fusion beyond B64.

No corresponding ANCE granularity projection was performed; therefore, no cross-backend claim about token-, Q/A-, or turn-level granularity is made.

### Teacher agreement on held-out development data

The original Teacher labels provide a common behavioral reference for the pretrained, imitation-control, and BM25-treatment checkpoints. Evaluating this agreement on TopiOCQA dev shows whether the learned selection behavior generalizes beyond the training population.

The comparison covers the 2,104 queries with at least two history turns because Teacher labels are unavailable for the remaining 410 queries. Predictions and labels are aligned only over history WordPieces visible within the selector’s 512-token input. All Teacher categories are collapsed to binary KEEP. KEEP precision, recall, and F1 measure which evidence is retained, while span metrics capture whether contiguous evidence regions are reproduced.

The imitation control continues learning the original Teacher targets, so its agreement measures additional Teacher imitation. The BM25 treatment learns corrected targets, so reduced agreement identifies the intended shift away from the original supervision. MRR subsequently determines whether this behavioral shift improves retrieval.

In [18]:
# Persistent Teacher-agreement evaluation on the labeled dev subset.
# The 205 queries without history and the 205 queries with exactly one
# history turn have no Teacher labels and are excluded from F1.
TEACHER_DEV_DATASET_SHA256 = (
    "43ced1a1f302672a29737d14fe02a1d9e"
    "6c751ea0eb1225aebe8222b72f49c37"
)
TEACHER_DEV_MANIFEST_SHA256 = (
    "548ad3eb185d6315ad9b359721cbeba21"
    "1005962a41d6f6b6bcc66ace5af742b"
)
TEACHER_DEV_PROTOCOL_HASH = (
    "1bb7fbf0e48741db4ff349d075940041f"
    "7c471d0ca46ab99db90b1b273cadcc9"
)
TEACHER_DEV_SEED = 42
teacher_dev_labels_path = artifact_path(
    "experiments/results/04_teacher"
    "/rocc_history_labels_topiocqa_dev.jsonl"
)
teacher_dev_source_manifest_path = artifact_path(
    "experiments/results/04_teacher"
    "/rocc_history_labels_topiocqa_dev.manifest.json"
)
teacher_dev_output_dir = RESULT_DIR / "teacher_dev_agreement"
teacher_dev_cache_paths = (
    teacher_dev_output_dir / "metrics.csv",
    teacher_dev_output_dir / "population.csv",
    teacher_dev_output_dir / "manifest.json",
)
teacher_source_presence = (
    teacher_dev_labels_path.is_file(),
    teacher_dev_source_manifest_path.is_file(),
)
if teacher_source_presence[0] != teacher_source_presence[1]:
    raise RuntimeError(
        "Teacher dev labels and source manifest are only partially "
        "present."
    )
teacher_cache_presence = tuple(
    path.is_file() for path in teacher_dev_cache_paths
)

if all(teacher_source_presence):
    teacher_identity = bind_notebook_lineage(
        RESULT_DIR, "teacher_dev_inputs", {
            "labels": TEACHER_DEV_DATASET_SHA256,
            "manifest": TEACHER_DEV_MANIFEST_SHA256,
        }, {"labels": teacher_dev_labels_path, "manifest": teacher_dev_source_manifest_path},
        manifest_fields=(("labels", teacher_dev_source_manifest_path, ("dataset_sha256",)),),
    )
    TEACHER_DEV_DATASET_SHA256 = teacher_identity["labels"]
    TEACHER_DEV_MANIFEST_SHA256 = teacher_identity["manifest"]

teacher_dev_agreement_result = None
teacher_dev_agreement_extension = None
teacher_dev_metrics = pd.DataFrame()
teacher_dev_population = pd.DataFrame()

if not any(teacher_source_presence) and not any(
    teacher_cache_presence
):
    display(
        Markdown(
            "**Teacher agreement:** Neither the persisted subartifact "
            "nor the Teacher source files exist in this artifact "
            "root. This backend-independent supplemental evaluation "
            "is skipped for the current run."
        )
    )
else:
    from experiments.rocc.teacher_dev_agreement import (
        load_or_compute_topiocqa_teacher_dev_agreement,
    )

    teacher_dev_agreement_result = (
        load_or_compute_topiocqa_teacher_dev_agreement(
            result_dir=RESULT_DIR,
            full_dev_rows=dev_rows,
            full_dev_population_sha256=population_sha256,
            checkpoints=CHECKPOINTS,
            expected_checkpoint_sha256=(
                EXPECTED_CHECKPOINT_HASHES
            ),
            checkpoint_manifests={
                "pretrained": PRETRAINED_MANIFEST,
                "imitation_control_e6": (
                    CONTROL_CHECKPOINT.parent / "manifest.json"
                ),
                "bm25_treatment_e6": (
                    TREATMENT_CHECKPOINT.parent / "manifest.json"
                ),
            },
            teacher_labels_path=teacher_dev_labels_path,
            teacher_manifest_path=(
                teacher_dev_source_manifest_path
            ),
            expected_teacher_dataset_sha256=(
                TEACHER_DEV_DATASET_SHA256
            ),
            expected_teacher_manifest_sha256=(
                TEACHER_DEV_MANIFEST_SHA256
            ),
            expected_teacher_protocol_hash=(
                TEACHER_DEV_PROTOCOL_HASH
            ),
            expected_teacher_seed=TEACHER_DEV_SEED,
            experiment_config=EXPERIMENT_CONFIG,
            encoding_cache_dir=(
                PROJECT_ROOT
                / ".cache/master_thesis/05_history_selector"
            ),
            device=DEVICE,
            batch_size=SELECTOR_BATCH_SIZE,
            progress=True,
        )
    )
    teacher_dev_metrics = teacher_dev_agreement_result.metrics
    teacher_dev_population = (
        teacher_dev_agreement_result.population
    )
    teacher_dev_agreement_extension = {
        "protocol": teacher_dev_agreement_result.manifest[
            "protocol"
        ],
        "scope": "result_dir",
        "manifest": str(
            teacher_dev_agreement_result.manifest_path.relative_to(
                RESULT_DIR
            )
        ),
        "manifest_sha256": sha256_file(
            teacher_dev_agreement_result.manifest_path
        ),
    }

    if REUSE_COMPLETE_ANALYSIS:
        updated_manifest = dict(existing_final_manifest)
        updated_extensions = dict(
            updated_manifest.get("extensions", {})
        )
        recorded_extension = updated_extensions.get(
            "teacher_dev_agreement"
        )
        if (
            recorded_extension is not None
            and recorded_extension
            != teacher_dev_agreement_extension
        ):
            raise RuntimeError(
                "Teacher-agreement extension differs in the "
                "main manifest."
            )
        if recorded_extension is None:
            updated_extensions["teacher_dev_agreement"] = (
                teacher_dev_agreement_extension
            )
            updated_manifest["extensions"] = updated_extensions
            write_json(backend_manifest_path, updated_manifest)
            existing_final_manifest = updated_manifest

    system_labels = {
        "pretrained": "NB05-Pretrained",
        "imitation_control_e6": "NB06-Control E6",
        "bm25_treatment_e6": "NB06-Treatment E6",
    }
    teacher_dev_display = teacher_dev_metrics[
        [
            "system",
            "n",
            "token_total",
            "token_accuracy",
            "keep_precision",
            "keep_recall",
            "keep_f1",
            "span_precision",
            "span_recall",
            "span_f1",
        ]
    ].copy()
    teacher_dev_display["system"] = teacher_dev_display[
        "system"
    ].map(system_labels)
    teacher_dev_display = teacher_dev_display.rename(
        columns={
            "system": "System",
            "n": "Queries",
            "token_total": "Visible history WordPieces",
            "token_accuracy": "Token Accuracy",
            "keep_precision": "KEEP Precision",
            "keep_recall": "KEEP Recall",
            "keep_f1": "KEEP F1",
            "span_precision": "Span Precision",
            "span_recall": "Span Recall",
            "span_f1": "Span F1",
        }
    )
    display(
        teacher_dev_display.style.format(
            {
                "Token Accuracy": "{:.4f}",
                "KEEP Precision": "{:.4f}",
                "KEEP Recall": "{:.4f}",
                "KEEP F1": "{:.4f}",
                "Span Precision": "{:.4f}",
                "Span Recall": "{:.4f}",
                "Span F1": "{:.4f}",
            }
        ).hide(axis="index")
    )


System,Queries,Visible history WordPieces,Token Accuracy,KEEP Precision,KEEP Recall,KEEP F1,Span Precision,Span Recall,Span F1
NB05-Pretrained,2104,393722,0.9119,0.5331,0.7432,0.6209,0.2675,0.5042,0.3495
NB06-Control E6,2104,393722,0.9194,0.5689,0.6818,0.6203,0.3478,0.4904,0.4069
NB06-Treatment E6,2104,393722,0.8743,0.4213,0.7600,0.5421,0.1216,0.4084,0.1874


**BM25.** Base, imitation-control and corrected-label `I+R+D` reach 0.1846, 0.1763 and 0.1799 MRR. Control minus base gives -0.0082 MRR, 95% CI [-0.0139, -0.0026]; treatment minus base gives -0.0047 MRR, 95% CI [-0.0112, +0.0018]; treatment minus control gives +0.0036 MRR, 95% CI [-0.0034, +0.0106]. For standalone `R`, the corresponding base/control/treatment means are 0.1598/0.1529/0.1602.

**ANCE.** Base, imitation-control and corrected-label `I+R+D` reach 0.2463, 0.2426 and 0.2435 MRR. Control minus base gives -0.0037 MRR, 95% CI [-0.0095, +0.0022]; treatment minus base gives -0.0028 MRR, 95% CI [-0.0100, +0.0046]; treatment minus control gives +0.0010 MRR, 95% CI [-0.0065, +0.0083]. For standalone `R`, the corresponding base/control/treatment means are 0.2440/0.2355/0.2364.

Teacher agreement and retrieval quality measure different outcomes. These paired intervals should be used to distinguish supported retrieval changes from differences in point estimates.

### Current-query-only IterCQR baseline

This baseline removes the complete conversation history before B64 IterCQR rewriting. It isolates how much retrieval quality is available from the current query alone and provides the history-free reference for the `I`, `R`, and fused routes. The persisted subartifact is reused only when its population, model, serializer, query bundle, ranking store, and metric hashes remain valid.

In [19]:
current_query_only_result = load_or_compute_current_query_only(
    result_dir=RESULT_DIR,
    dataset="topiocqa",
    samples=dev_samples,
    sample_ids=sample_ids,
    gold_by_sample=gold_by_sample,
    population_sha256=population_sha256,
    model_dir=ITERCQR_MODEL_DIR,
    expected_model_sha256=(
        query_bundle_manifest["itercqr_model_sha256"]
    ),
    expected_serializer_sha256=(
        query_bundle_manifest["serializer_sha256"]
    ),
    pipeline_config=PIPELINE_CONFIG,
    backend=BACKEND,
    bm25_index_dir=BM25_INDEX_DIR,
    bm25_k1=0.9,
    bm25_b=0.4,
    dense_index_dir=DENSE_INDEX_DIR,
    azure_sas_file=None,
    device=str(DEVICE),
    budget=PRIMARY_BUDGET,
    top_k=RETRIEVAL_TOP_K,
    eval_ks=EVAL_KS,
    retrieval_batch_size=RETRIEVAL_BATCH_SIZE,
    retrieval_workers=PIPELINE_CONFIG.retrieval_workers,
    progress=True,
)
if not current_query_only_result.ready:
    raise RuntimeError(
        "TopiOCQA current-query-only evaluation is incomplete."
    )
current_query_only_summary = current_query_only_result.summary
current_query_only_extension = (
    current_query_only_result.extension
)
display(current_query_only_summary.round(4))

if REUSE_COMPLETE_ANALYSIS:
    updated_manifest = dict(existing_final_manifest)
    updated_extensions = dict(
        updated_manifest.get("extensions", {})
    )
    recorded_extension = updated_extensions.get(
        "current_query_only"
    )
    if (
        recorded_extension is not None
        and recorded_extension != current_query_only_extension
    ):
        raise RuntimeError(
            "Current-query-only extension differs in the "
            "main manifest."
        )
    if recorded_extension is None:
        updated_extensions["current_query_only"] = (
            current_query_only_extension
        )
        updated_manifest["extensions"] = updated_extensions
        write_json(backend_manifest_path, updated_manifest)
        existing_final_manifest = updated_manifest


,system,arm,budget,n,MRR,nDCG@3,R@10,R@100,R@1000,mean_input_length,max_input_length
0,itercqr,CurrentQueryOnly,64,2514,0.0351,0.0325,0.0581,0.0927,0.146,10.8067,30


### Final manifest

The manifest binds the population, query bundle, checkpoints, protocol, backend, and result files through SHA-256. This enables verified reuse and detects experimental drift. `selection_performed=false` and the prohibition of `decision.json` preserve TopiOCQA dev exclusively for final evaluation.

In [20]:
if REUSE_COMPLETE_ANALYSIS:
    final_manifest = dict(existing_final_manifest)
    if any(path.name == "decision.json" for path in RESULT_DIR.rglob("*")):
        raise RuntimeError("NB07a may not write decision.json.")
    display(Markdown(
        f"**NB07a restored ({BACKEND.upper()}):** "
        f"{len(sample_ids):,} queries, query bundle "
        f"{bundle_sha256}, no recomputation."
    ))
else:
    result_files = sorted(
        path
        for path in backend_dir.rglob("*")
        if (
            path.is_file()
            and path.name != "manifest.json"
            and "latency" not in path.name.lower()
        )
    )
    final_manifest = {
        "schema_version": 1,
        "protocol": "nb07a_topiocqa_final_v2",
        "d_no_history_query_rule": D_NO_HISTORY_QUERY_RULE,
        "backend": BACKEND,
        "population_sha256": population_sha256,
        "population_queries": len(sample_ids),
        "query_bundle_sha256": bundle_sha256,
        "reconstructed_query_bundle_sha256": (
            reconstructed_bundle_sha256
        ),
        "reconstruction_matches_bundle": (
            reconstruction_matches_bundle
        ),
        "checkpoint_sha256": EXPECTED_CHECKPOINT_HASHES,
        "budgets": list(BUDGETS),
        "primary_budget": PRIMARY_BUDGET,
        "rrf_k": RRF_K,
        "retrieval_top_k": RETRIEVAL_TOP_K,
        "bootstrap_replicates": BOOTSTRAP_REPLICATES,
        "primary_comparison_order": list(PRIMARY_COMPARISON_ORDER),
        "sampling_seed_scheme": SAMPLING_SEED_SCHEME,
        "software": software_manifest,
        "backend_identity": backend_identity,
        "files": {
            str(path.relative_to(backend_dir)): sha256_file(path)
            for path in result_files
        },
        "extensions": {
            **(
                {
                    "granularity_projection": (
                        granularity_projection_extension
                    )
                }
                if granularity_projection_extension is not None
                else {}
            ),
            **(
                {
                    "teacher_dev_agreement": (
                        teacher_dev_agreement_extension
                    )
                }
                if teacher_dev_agreement_extension is not None
                else {}
            ),
            **(
                {
                    "current_query_only": (
                        current_query_only_extension
                    )
                }
                if current_query_only_extension is not None
                else {}
            ),
        },
        "complete": True,
        "selection_performed": False,
    }
    write_json(backend_dir / "manifest.json", final_manifest)

    if any(path.name == "decision.json" for path in RESULT_DIR.rglob("*")):
        raise RuntimeError("NB07a may not write decision.json.")
    display(
        Markdown(
            f"**NB07a completed ({BACKEND.upper()}):** "
            f"{len(sample_ids):,} queries, query bundle `{bundle_sha256}`, "
            "no selection on dev."
        )
    )


**NB07a completed (ANCE):** 2,514 queries, query bundle `c7358e2430302918d218f41e0d55ffb8a1991773872aadb39e85252008ed28e7`, no selection on dev.

## Conclusion from the frozen final results

**BM25.** At B64, the base selector gives 0.1446 MRR for `I`, 0.1598 for `R`, 0.1570 for `D`, and 0.1846 for `I+R+D`. IterCQR-G+2S reaches 0.1532. The final fusion therefore exceeds the matched-budget sampled-rewrite baseline by 0.0314 MRR. Treatment gives 0.1602 for `R` and 0.1799 for `I+R+D`, compared with 0.1529 and 0.1763 for the imitation control.

**ANCE.** At B64, the base selector gives 0.2116 MRR for `I`, 0.2440 for `R`, 0.1741 for `D`, and 0.2463 for `I+R+D`. IterCQR-G+2S reaches 0.2154. The final fusion therefore exceeds the matched-budget sampled-rewrite baseline by 0.0309 MRR. Treatment gives 0.2364 for `R` and 0.2435 for `I+R+D`, compared with 0.2355 and 0.2426 for the imitation control.

These figures come from the frozen `results/07a_topiocqa_final/backends/{bm25,ance}/` route summaries and paired comparisons. The final result tables show both backends separately. The older experiment cells retain the backend of their last execution.

**BM25.** Base B64 `I+R+D` versus the single B64 IterCQR rewrite gives +0.0400 MRR, 95% CI [+0.0314, +0.0486]. The comparison with B64 IterCQR-G+2S gives +0.0314 MRR, 95% CI [+0.0232, +0.0400]. Against B512 IterCQR-G+2S, the B64 fusion gives +0.0037 MRR, 95% CI [-0.0050, +0.0125]. These are paired comparisons on the same queries; a positive interval excludes zero in favor of the left-hand system.

**ANCE.** Base B64 `I+R+D` versus the single B64 IterCQR rewrite gives +0.0347 MRR, 95% CI [+0.0251, +0.0445]. The comparison with B64 IterCQR-G+2S gives +0.0309 MRR, 95% CI [+0.0213, +0.0409]. Against B512 IterCQR-G+2S, the B64 fusion gives -0.0161 MRR, 95% CI [-0.0263, -0.0064]. These are paired comparisons on the same queries; a positive interval excludes zero in favor of the left-hand system.

**BM25 coverage.** Base `I` has $\mathrm{Recall}@1000$=0.6130; `I+R+D` reaches 0.8027. These values describe the final I/R/D fusion, not Viterbi-plus-FFBS fusion within route R.

**ANCE coverage.** Base `I` has $\mathrm{Recall}@1000$=0.6146; `I+R+D` reaches 0.7733. These values describe the final I/R/D fusion, not Viterbi-plus-FFBS fusion within route R.

**BM25.** Without history, `I` and `R` have the same input and score 0.1564 MRR. This does not imply equality with direct retrieval or rank fusion: `I+R+D` scores 0.1685. At depths 2–4, base fusion exceeds the single IterCQR rewrite by +0.0396 MRR. At depths 5–6, base fusion exceeds the single IterCQR rewrite by +0.0567 MRR. At depths 7–8, base fusion exceeds the single IterCQR rewrite by +0.0630 MRR.

**ANCE.** Without history, `I` and `R` have the same input and score 0.3020 MRR. This does not imply equality with direct retrieval or rank fusion: `I+R+D` scores 0.3008. At depths 2–4, base fusion exceeds the single IterCQR rewrite by +0.0401 MRR. At depths 5–6, base fusion exceeds the single IterCQR rewrite by +0.0317 MRR. At depths 7–8, base fusion exceeds the single IterCQR rewrite by +0.0400 MRR.

**BM25.** Base, imitation-control and corrected-label `I+R+D` reach 0.1846, 0.1763 and 0.1799 MRR. Control minus base gives -0.0082 MRR, 95% CI [-0.0139, -0.0026]; treatment minus base gives -0.0047 MRR, 95% CI [-0.0112, +0.0018]; treatment minus control gives +0.0036 MRR, 95% CI [-0.0034, +0.0106]. For standalone `R`, the corresponding base/control/treatment means are 0.1598/0.1529/0.1602.

**ANCE.** Base, imitation-control and corrected-label `I+R+D` reach 0.2463, 0.2426 and 0.2435 MRR. Control minus base gives -0.0037 MRR, 95% CI [-0.0095, +0.0022]; treatment minus base gives -0.0028 MRR, 95% CI [-0.0100, +0.0046]; treatment minus control gives +0.0010 MRR, 95% CI [-0.0065, +0.0083]. For standalone `R`, the corresponding base/control/treatment means are 0.2440/0.2355/0.2364.

Teacher agreement and retrieval quality measure different outcomes. These paired intervals should be used to distinguish supported retrieval changes from differences in point estimates.

## Result summary

These tables read the saved results of this notebook and display them directly, without a separate export. Additional cross-experiment analyses appear after the corresponding supporting scripts have been run; unavailable analyses are identified explicitly.


In [6]:
from experiments.scripts.analysis_results import display_results

display_results(PROJECT_ROOT / "experiments", '07a_topiocqa_final_retrieval')


Additional retrieval comparisons are not available yet: <project>/experiments/results/10_efficiency_analysis/figures/data/results_retrieval_paired_comparisons.csv. Run the README's teacher/figure analysis scripts and rerun this cell. Showing the completed notebook results below.


**TopiOCQA Dev / BM25: route means**

,system,budget,arm,n,MRR,nDCG@3,R@10,R@100,R@1000
0,bm25_treatment_e6,64,D,2514,0.151847,0.135687,0.262928,0.493636,0.727128
1,bm25_treatment_e6,64,I,2514,0.144571,0.134408,0.245823,0.431185,0.612967
2,bm25_treatment_e6,64,I+D,2514,0.169395,0.153866,0.291567,0.542164,0.764519
3,bm25_treatment_e6,64,I+R,2514,0.170436,0.154126,0.305091,0.537788,0.759745
4,bm25_treatment_e6,64,I+R+D,2514,0.179906,0.161565,0.314638,0.575179,0.799125
5,bm25_treatment_e6,64,R,2514,0.160250,0.143628,0.285203,0.517900,0.737072
6,bm25_treatment_e6,64,R+D,2514,0.170375,0.151541,0.297932,0.559268,0.784407
7,bm25_treatment_e6,128,D,2514,0.152199,0.136932,0.261734,0.496420,0.735481
8,bm25_treatment_e6,128,I,2514,0.157267,0.142127,0.276054,0.497613,0.710819
9,bm25_treatment_e6,128,I+D,2514,0.174305,0.157190,0.303500,0.574781,0.794352


**TopiOCQA Dev / BM25: paired comparisons**

,comparison,metric,left_system,left_budget,left_arm,right_system,right_budget,right_arm,n,delta,ci95_low,ci95_high,seed,replicates
0,pretrained_ird_minus_itercqr1,MRR,pretrained,64,I+R+D,itercqr,64,IterCQR-1,2514,0.039997,0.031420,0.048585,1000,10000
1,pretrained_ird_minus_itercqr1,nDCG@3,pretrained,64,I+R+D,itercqr,64,IterCQR-1,2514,0.031670,0.022043,0.041165,1001,10000
2,pretrained_ird_minus_itercqr1,R@10,pretrained,64,I+R+D,itercqr,64,IterCQR-1,2514,0.073986,0.060859,0.087510,1002,10000
3,pretrained_ird_minus_itercqr1,R@100,pretrained,64,I+R+D,itercqr,64,IterCQR-1,2514,0.155927,0.139618,0.171838,1003,10000
4,pretrained_ird_minus_itercqr1,R@1000,pretrained,64,I+R+D,itercqr,64,IterCQR-1,2514,0.189737,0.173429,0.206454,1004,10000
5,pretrained_ird_minus_itercqr_g2s,MRR,pretrained,64,I+R+D,itercqr,64,IterCQR-G+2S-RRF10,2514,0.031402,0.023151,0.039986,1005,10000
6,pretrained_ird_minus_itercqr_g2s,nDCG@3,pretrained,64,I+R+D,itercqr,64,IterCQR-G+2S-RRF10,2514,0.025528,0.016261,0.034947,1006,10000
7,pretrained_ird_minus_itercqr_g2s,R@10,pretrained,64,I+R+D,itercqr,64,IterCQR-G+2S-RRF10,2514,0.058473,0.044948,0.072395,1007,10000
8,pretrained_ird_minus_itercqr_g2s,R@100,pretrained,64,I+R+D,itercqr,64,IterCQR-G+2S-RRF10,2514,0.129276,0.113365,0.145187,1008,10000
9,pretrained_ird_minus_itercqr_g2s,R@1000,pretrained,64,I+R+D,itercqr,64,IterCQR-G+2S-RRF10,2514,0.159905,0.143596,0.176213,1009,10000


...

,route,n,mean_tokens,maximum_tokens
0,I512,2514,150.091090,NaN
1,I64,2514,56.656722,NaN
2,R512,2514,41.875497,160.0
3,R64,2514,40.889419,NaN


**input length counts**

,condition,n,queries,percent
0,I64 shorter than I512,2514,1914,76.133652
1,R512 above64 through128,2514,239,9.506762
2,R512 above128,2514,1,0.039777
3,R512 above64,2514,240,9.546539


**input length histogram**

,tokens_from,tokens_through,I512,I64,R512
0,1,8,7,7,7
1,9,16,144,144,151
2,17,24,67,67,213
3,25,32,97,97,375
4,33,40,68,68,506
5,41,48,68,68,453
6,49,56,75,75,341
7,57,64,74,1988,228
8,65,72,76,0,140
9,73,80,78,0,51
